## IC的p-value的意义和计算方法

### p-value的意义 (The Meaning)

p-value（p值）是用来进行**假设检验**的，它回答了一个核心问题：“我们观察到的结果，有多大的可能性是纯粹由随机运气造成的？”

在IC值的上下文中：

  * **零假设 ($H\_0$)**: 我们的因子（模型预测值）与未来的真实收益率之间 **没有任何关系**。它们是完全不相关的，任何计算出来的非零IC值都只是随机的巧合。
  * **备择假设 ($H\_a$)**: 我们的因子与未来的真实收益率之间 **确实存在关系**（无论是正相关还是负相关）。

**p-value的精确定义是：**

> 假设“零假设”为真（即因子无效），我们观察到当前这样大小的IC值，或者更极端IC值的概率。

**通俗的解释：**

  * **p-value很小 (例如, $p < 0.05$)**: 这意味着，如果你的模型真的只是一个“随机猜测器”，那么得到你现在这个IC值的概率非常非常低（低于5%）。因此，你就有很强的信心拒绝“零假设”，认为你的模型确实具有统计上显著的预测能力。这个IC值很可能是“真材实料”，而非运气。
  * **p-value很大 (例如, $p > 0.05$)**: 这意味着，即使你的模型是个“随机猜测器”，也有很大的可能性（大于5%）得到你现在这个IC值。因此，你没有足够的证据拒绝“零假设”，你不能断定你的模型有效。这个IC值很可能只是噪音或运气。


### p-value的计算方法 (The Calculation)

你的代码中使用了`scipy.stats.spearmanr`来计算Spearman相关系数（IC值）和p-value。这个计算过程在统计学上是标准化的。

对于Spearman相关系数 $rankic$，其p-value的计算通常基于一个与t分布相关的检验统计量。

1.  **计算检验统计量 $t$**:
    该统计量衡量了我们计算出的IC值偏离0的程度。其计算公式为：
    $$t = r_s \sqrt{\frac{n-2}{1 - r_s^2}}$$
    其中：

      * $r\_s$ 是你计算出的Spearman IC值(rankic)。
      * $n$ 是样本数量（即数据点的个数）。

2.  **从t分布中查找p-value**:
    这个计算出的 $t$ 值服从一个自由度为 $n-2$ 的**t分布**。然后，通过查询t分布的累积分布函数（CDF），就可以得到p-value。这个p-value代表了在t分布中，获得比我们计算出的 $|t|$ 值更大或相等的绝对值的概率。

**直观理解这个公式：**

  * **IC值 ($r\_s$) 越大**：$t$ 值就越大，p-value就越小。这很合理，因为更强的相关性更不可能是巧合。
  * **样本量 ($n$) 越大**：$t$ 值也越大，p-value就越小。这也非常合理，因为在更多数据上验证的IC值可信度更高。即使是一个较小的IC值，如果是在海量数据上得到的，也可能是显著的。

`scipy`库为你自动完成了所有这些复杂的计算，你只需要调用函数即可。

-----


## 各fold测试集Rankic平均值和拼接测试集Rankic的差异

### 核心关键：模型的“刻度尺”在变化

请先接受一个核心设定，这是真实世界中模型运作的方式：
> 模型输出的预测分，其“意义”或“刻度”，在不同的市场环境下是会变化的。例如，在熊市中，一个 `+0.01` 的预测分可能已经是“极度看涨”的信号了；但在牛市中，`+0.01` 可能只是一个“中性偏弱”的信号。

您的模型正在尝试学习非常复杂的规律，它无法做到完美地用一套固定的刻度来衡量所有情况。于是，**不同市场环境（Folds）下的预测分，它们的分布区间就会发生重叠**。

### 一个严谨且符合您所有要求的例子

* **Fold 1 (牛市)**: 预测范围 `[0.02, 0.05]`，真实收益范围 `[2%, 4%]`
* **Fold 2 (熊市)**: 预测范围 `[-0.02, 0.01]`，真实收益范围 `[-3%, -1%]`

在各自的Fold内部，我们假设模型的排序能力都是完美的。

**Fold 1 (牛市):**
| 股票 | 模型预测分 | 真实收益率 |
| :--- | :--- | :--- |
| A | 0.05 | 4% |
| B | 0.02 | 2% |
> 局部Rank IC = +1.0

**Fold 2 (熊市):**
| 股票 | 模型预测分 | 真实收益率 |
| :--- | :--- | :--- |
| X | 0.01 | -1% |
| Y | -0.02 | -3% |
> 局部Rank IC = +1.0

**Rank IC 平均值 = (1.0 + 1.0) / 2 = +1.0**。 

**现在，我们把这4个数据点拼接起来，进行全局排名。**

**拼接后的数据，并按“模型预测分”从高到低排序：**
| 股票 | 模型预测分 | 真实收益率 |
| :--- | :--- | :--- |
| A | 0.05 | 4% |
| B | 0.02 | 2% |
| X | 0.01 | -1% |
| Y | -0.02 | -3% |

**计算全新的“全局排名”：**
| 股票 | **全局**预测排名 | **全局**真实排名 |
| :--- | :--- | :--- |
| A | **1** | A (4%) | **1** |
| B | **2** | B (2%) | **2** |
| X | **3** | X (-1%) | **3** |
| Y | **4** | Y (-3%) | **4** |

在这个例子中，拼接后的Rank IC依然是+1.0。为什么？因为牛市的预测分区间和熊市的预测分区间是**完全分离**的，牛市中“最差”的预测分(0.02)也比熊市中“最好”的(0.01)要高。**这恰恰就是您脑海中的直觉，也是您提出疑问的来源。**

---

### 当预测分的“刻度尺”发生重叠时

现在，我们来看一种更真实的情况。模型无法完美地把不同环境下的预测分区间完全分离开。

我们对上面的例子做一个微小的、但至关重要的改动，**让两个Fold的预测分区间出现重叠**，并且所有数值都仍在您给定的范围内。

**Fold 1 (牛市):** (真实收益不变)
| 股票 | 模型预测分 | 真实收益率 |
| :--- | :--- | :--- |
| A | 0.05 | 4% |
| B | **0.005** | 2% | <-- B的预测分调低了
> 局部Rank IC 依然是 +1.0 (因为0.05 > 0.005)

**Fold 2 (熊市):** (真实收益不变)
| 股票 | 模型预测分 | 真实收益率 |
| :--- | :--- | :--- |
| X | **0.01** | -1% | <-- X的预测分不变，但现在比B的预测分高了
| Y | -0.02 | -3% |
> 局部Rank IC 依然是 +1.0 (因为0.01 > -0.02)

**Rank IC 平均值** 依然是完美无瑕的 `+1.0`。

**现在，让我们再次拼接数据，并按“模型预测分”从高到低排序：**
| 股票 | 模型预测分 | 真实收益率 |
| :--- | :--- | :--- |
| A | 0.05 | 4% |
| X | **0.01** | **-1%** |
| B | **0.005** | **2%** |
| Y | -0.02 | -3% |

**我们来计算这个新组合的全局排名：**
| 股票 | **全局**预测排名 | **全局**真实排名 |
| :--- | :--- | :--- |
| A (预测分最高) | **1** | A (收益率4%最高) | **1** |
| X (预测分第二) | **2** | B (收益率2%第二) | **2** |
| B (预测分第三) | **3** | X (收益率-1%第三) | **3** |
| Y (预测分第四) | **4** | Y (收益率-3%第四) | **4** |

**最终用于计算拼接后Rank IC的两个序列是：**
* 全局预测排名序列: `[1, 2, 3, 4]`
* 全局真实排名序列: `[1, 3, 2, 4]` 

**这个冲突是怎么产生的？**
* 在全局预测排名中，股票X排第2，股票B排第3。
* 但在全局真实排名中，股票B（收益2%）的排名（第2）却高于股票X（收益-1%）的排名（第3）。

这就是**“排名倒挂”**。这个倒挂不是因为一两个“极端值”，而是因为两个Fold的预测分和真实收益率的**整体分布（或者说对应关系）发生了系统性的错位**。

### 最终结论

1.  **这不是少数极端值的问题**：在我们的新例子里，没有任何一个数据点是“极端值”，所有的值都在合理的范围内。然而，仅仅因为**预测分的“重叠”**，就足以导致全局排名出现混乱。在拥有成百上千个数据点的真实世界里，这种重叠和交错会发生无数次，其累积效应就会造成拼接后的Rank IC大幅下降。

2.  **问题的根源**：是模型的 **“刻度尺”不统一** 。模型给出的 `0.01` 这个分数，在熊市（Fold 2）里代表了最好的投资机会，但在全局视角下，它却对应了一个负的收益率，远不如牛市（Fold 1）里一个仅被评为 `0.005` 分的投资机会。

3.  **Rankic**：它是一个度量工具，衡量了您的模型在多大程度上混淆了不同市场环境下的“刻度”。差距越大，说明模型区分不同市场环境的能力越弱，其预测分的“绝对值”就越不可靠（尽管其在“局部”的排序能力可能很强）。

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import warnings
from typing import List, Tuple, Dict
warnings.filterwarnings('ignore')

# ================================
# 参数配置区域
# ================================
TICKER = 'QQQ'
START_DATE = '2009-01-01'
END_DATE = '2025-06-30'
OUTPUT_DIR = 'QQQ_Rolling_Validation'
OUTPUT_FILENAME = 'QQQ_rolling_validation_dataset.xlsx'

# 滚动验证参数
INITIAL_TRAIN_YEARS = 5  # 初始训练集长度（年）
TEST_PERIOD_YEARS = 1    # 每个测试窗口长度（年）
PURGE_DAYS = 10         # 数据清洗交易日数（避免标签泄露）
TARGET_PREDICTION_DAYS = 10  # 预测目标天数

# 特征工程时间窗口
SHORT_TERM_MOMENTUM = [2, 3, 5, 7, 10]
SHORT_MID_TERM_TREND = [14, 20, 30, 50]
MID_LONG_TERM_TREND = [75, 100, 150, 200, 250, 300, 350, 400, 450, 500]
TARGET_HORIZONS = [5, 10, 20]

# ================================
# 核心功能函数
# ================================

def get_all_time_windows() -> List[int]:
    """获取所有时间窗口的排序列表"""
    all_windows = SHORT_TERM_MOMENTUM + SHORT_MID_TERM_TREND + MID_LONG_TERM_TREND
    return sorted(list(set(all_windows)))

def generate_rolling_folds() -> List[Tuple[str, str, str, str]]:
    """
    生成滚动验证的时间窗口
    
    Returns:
        List[Tuple]: [(train_start, train_end, test_start, test_end), ...]
    """
    folds = []
    
    # 起始点
    train_start = pd.to_datetime(START_DATE)
    
    # 第一个训练集结束点（初始5年）
    first_train_end = train_start + pd.DateOffset(years=INITIAL_TRAIN_YEARS) - timedelta(days=1)
    
    # 第一个测试集开始点
    current_test_start = first_train_end + timedelta(days=1)
    current_train_end = first_train_end
    
    end_date = pd.to_datetime(END_DATE)
    fold_num = 1
    
    while current_test_start < end_date:
        # 计算当前测试集结束点
        current_test_end = min(
            current_test_start + pd.DateOffset(years=TEST_PERIOD_YEARS) - timedelta(days=1),
            end_date
        )
        
        # 确保测试集至少有6个月的数据
        if (current_test_end - current_test_start).days < 180:
            break
            
        fold_info = (
            train_start.strftime('%Y-%m-%d'),
            current_train_end.strftime('%Y-%m-%d'), 
            current_test_start.strftime('%Y-%m-%d'),
            current_test_end.strftime('%Y-%m-%d')
        )
        
        folds.append(fold_info)
        
        print(f"Fold {fold_num}: Train({fold_info[0]} to {fold_info[1]}) -> Test({fold_info[2]} to {fold_info[3]})")
        
        # 为下一个fold准备
        current_train_end = current_test_end  # 扩展窗口：训练集包含之前所有数据
        current_test_start = current_test_end + timedelta(days=1)
        fold_num += 1
    
    print(f"\n总共生成 {len(folds)} 个滚动验证窗口")
    return folds

def get_extended_data() -> pd.DataFrame:
    """获取扩展的股票数据，用于特征工程"""
    print(f"获取 {TICKER} 扩展数据...")
    
    all_windows = get_all_time_windows()
    max_window = max(all_windows)
    max_horizon = max(TARGET_HORIZONS)
    
    # 扩展时间范围以计算指标
    lookback_days = int(max_window * 2 + 100)  # 更保守的回看天数
    forward_days = max_horizon + 50
    
    start_dt = pd.to_datetime(START_DATE) - timedelta(days=lookback_days)
    end_dt = pd.to_datetime(END_DATE) + timedelta(days=forward_days)
    
    # 获取股票数据
    df = yf.Ticker(TICKER).history(
        start=start_dt.strftime('%Y-%m-%d'),
        end=end_dt.strftime('%Y-%m-%d'),
        auto_adjust=False,
        actions=False
    )
    
    if df.empty:
        raise ValueError(f"无法获取 {TICKER} 的数据")
    
    # 数据预处理
    df.index = df.index.tz_localize(None)
    df.rename(columns={
        'Open': 'open', 'High': 'high', 'Low': 'low',
        'Close': 'close', 'Volume': 'volume'
    }, inplace=True)
    
    df = df[['open', 'high', 'low', 'close', 'volume']]
    print(f"获取到 {len(df)} 条原始数据记录")
    
    return df

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """生成所有技术指标特征（保持原始特征工程）"""
    print("开始特征工程...")
    
    all_windows = get_all_time_windows()
    feature_df = df.copy()
    
    # 基础特征
    feature_df['turnover'] = feature_df['close'] * feature_df['volume']
    feature_df['daily_return'] = feature_df['close'].pct_change()
    
    # 原始特征工程
    for n in all_windows:
        feature_df[f'return_{n}d'] = feature_df['close'].pct_change(periods=n)
        feature_df[f'mean_return_{n}d'] = feature_df['daily_return'].rolling(window=n).mean()
        feature_df[f'volatility_{n}d'] = feature_df['daily_return'].rolling(window=n).std()
        if n >= 3:
            feature_df[f'skewness_{n}d'] = feature_df['daily_return'].rolling(window=n).skew()
        if n >= 4:
            feature_df[f'kurtosis_{n}d'] = feature_df['daily_return'].rolling(window=n).kurt()
        rolling_mean = feature_df['daily_return'].rolling(window=n).mean()
        rolling_std = feature_df['daily_return'].rolling(window=n).std()
        feature_df[f'sharpe_{n}d'] = rolling_mean / rolling_std
    
    print(f"特征工程完成，共生成 {feature_df.shape[1]} 个特征")
    return feature_df

def generate_target_variables(df: pd.DataFrame) -> pd.DataFrame:
    """生成目标变量（保持原始逻辑）"""
    print("生成目标变量...")
    target_df = df.copy()
    
    for horizon in TARGET_HORIZONS:
        target_df[f'y_ret_{horizon}d'] = target_df['close'].pct_change(periods=horizon).shift(-horizon)
        if horizon >= 10:
            future_returns = target_df['daily_return'].shift(-1)
            future_mean = future_returns.rolling(window=horizon).mean().shift(-(horizon-1))
            future_std = future_returns.rolling(window=horizon).std().shift(-(horizon-1))
            target_df[f'y_sharpe_{horizon}d'] = future_mean / future_std
    
    print(f"目标变量生成完成")
    return target_df

def apply_purging(full_df: pd.DataFrame, train_end_date: str, test_start_date: str, purge_trading_days: int = PURGE_DAYS) -> str:
    """
    应用数据清洗，从训练集末尾移除指定交易日数的数据
    
    Args:
        full_df: 完整数据集（用于获取交易日历）
        train_end_date: 原始训练集结束日期
        test_start_date: 测试集开始日期  
        purge_trading_days: 需要清洗的交易日数
    
    Returns:
        purged_train_end_date: 清洗后的训练集结束日期
    """
    train_end = pd.to_datetime(train_end_date)
    
    # 获取训练集结束日期之前的交易日
    trading_days_before_end = full_df[full_df.index <= train_end].index
    
    if len(trading_days_before_end) < purge_trading_days:
        print(f"警告：可用交易日不足{purge_trading_days}天，使用所有可用交易日")
        purge_trading_days = len(trading_days_before_end) - 1
    
    # 向前推purge_trading_days个交易日
    purged_train_end = trading_days_before_end[-(purge_trading_days + 1)]
    
    print(f"数据清洗：训练集从 {train_end_date} 缩减到 {purged_train_end.strftime('%Y-%m-%d')} (清洗{purge_trading_days}个交易日)")
    return purged_train_end.strftime('%Y-%m-%d')

def prepare_fold_data(full_df: pd.DataFrame, fold_info: Tuple[str, str, str, str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    为单个fold准备训练和测试数据
    
    Args:
        full_df: 完整的特征数据
        fold_info: (train_start, train_end, test_start, test_end)
    
    Returns:
        train_df, test_df: 清洗后的训练集和测试集
    """
    train_start, train_end, test_start, test_end = fold_info
    
    # 应用交易日数据清洗
    purged_train_end = apply_purging(full_df, train_end, test_start)
    
    # 提取数据
    train_df = full_df[
        (full_df.index >= pd.to_datetime(train_start)) & 
        (full_df.index <= pd.to_datetime(purged_train_end))
    ].copy()
    
    test_df = full_df[
        (full_df.index >= pd.to_datetime(test_start)) & 
        (full_df.index <= pd.to_datetime(test_end))
    ].copy()
    
    return train_df, test_df

def convert_to_chinese_names(df: pd.DataFrame) -> pd.DataFrame:
    """将列名转换为中文（保持原始逻辑）"""
    chinese_map = {
        'open': '开盘价', 'high': '最高价', 'low': '最低价', 'close': '收盘价', 
        'volume': '成交量', 'turnover': '成交额', 'daily_return': '日收益率'
    }
    
    for col in df.columns:
        if col in chinese_map:
            continue
        elif col.startswith('return_'):
            chinese_map[col] = f"{col.split('_')[1]}收益率"
        elif col.startswith('mean_return_'):
            chinese_map[col] = f"{col.split('_')[2]}平均收益"
        elif col.startswith('volatility_'):
            chinese_map[col] = f"{col.split('_')[1]}波动率"
        elif col.startswith('skewness_'):
            chinese_map[col] = f"{col.split('_')[1]}偏度"
        elif col.startswith('kurtosis_'):
            chinese_map[col] = f"{col.split('_')[1]}峰度"
        elif col.startswith('sharpe_'):
            chinese_map[col] = f"{col.split('_')[1]}夏普比率"
        elif col.startswith('y_ret_'):
            chinese_map[col] = f"目标_未来{col.split('_')[2]}回报率"
        elif col.startswith('y_sharpe_'):
            chinese_map[col] = f"目标_未来{col.split('_')[2]}夏普比率"
    
    return df.rename(columns=chinese_map)

def export_rolling_validation_data(full_df: pd.DataFrame, rolling_folds: List[Tuple]) -> str:
    """导出所有fold的滚动验证数据集"""
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    
    output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)
    
    # 识别特征列和目标列
    basic_cols = ['open', 'high', 'low', 'close', 'volume', 'turnover', 'daily_return']
    target_cols = [col for col in full_df.columns if col.startswith('y_')]
    feature_cols = [col for col in full_df.columns if col not in basic_cols + target_cols]
    
    print(f"基础列: {len(basic_cols)}, 特征列: {len(feature_cols)}, 目标列: {len(target_cols)}")
    
    # 转换为中文列名
    full_df_chinese = convert_to_chinese_names(full_df)
    
    # 导出到Excel
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        # 导出完整数据集（用于理解数据结构）
        sample_df = full_df_chinese[
            (full_df_chinese.index >= pd.to_datetime(START_DATE)) & 
            (full_df_chinese.index <= pd.to_datetime(END_DATE))
        ].dropna()
        
        sample_df.index = sample_df.index.strftime('%Y-%m-%d')
        sample_df.to_excel(writer, sheet_name='完整数据集', index=True, index_label='日期')
        
        # 导出所有fold的训练和测试数据
        successful_folds = 0
        for i, fold_info in enumerate(rolling_folds, 1):
            try:
                train_df, test_df = prepare_fold_data(full_df, fold_info)
                
                # 清理数据
                train_clean = train_df.dropna(subset=feature_cols + target_cols)
                test_clean = test_df.dropna(subset=feature_cols + target_cols)
                
                if len(train_clean) > 0 and len(test_clean) > 0:
                    # 转换为中文并格式化
                    train_chinese = convert_to_chinese_names(train_clean)
                    test_chinese = convert_to_chinese_names(test_clean)
                    
                    train_chinese.index = train_chinese.index.strftime('%Y-%m-%d')
                    test_chinese.index = test_chinese.index.strftime('%Y-%m-%d')
                    
                    # 导出 - 使用两位数编号确保排序
                    train_sheet_name = f'Fold{i:02d}_训练集'
                    test_sheet_name = f'Fold{i:02d}_测试集'
                    
                    train_chinese.to_excel(writer, sheet_name=train_sheet_name, index=True, index_label='日期')
                    test_chinese.to_excel(writer, sheet_name=test_sheet_name, index=True, index_label='日期')
                    
                    successful_folds += 1
                    print(f"Fold {i} 导出成功: 训练集 {len(train_clean)} 行, 测试集 {len(test_clean)} 行")
                else:
                    print(f"Fold {i} 跳过: 数据不足")
                    
            except Exception as e:
                print(f"Fold {i} 导出失败: {e}")
                continue
        
        # 导出fold信息摘要
        fold_summary = pd.DataFrame(rolling_folds, columns=['训练开始', '训练结束', '测试开始', '测试结束'])
        fold_summary['Fold编号'] = range(1, len(rolling_folds) + 1)
        fold_summary = fold_summary[['Fold编号', '训练开始', '训练结束', '测试开始', '测试结束']]
        fold_summary.to_excel(writer, sheet_name='Fold信息摘要', index=False)
    
    print(f"滚动验证数据集已导出: {output_path}")
    print(f"成功导出 {successful_folds}/{len(rolling_folds)} 个fold")
    return output_path

def main():
    """主函数：执行完整的滚动验证数据收集流程"""
    try:
        print("=" * 50)
        print("QQQ 滚动验证数据收集系统启动")
        print("=" * 50)
        
        # 1. 生成滚动验证窗口
        rolling_folds = generate_rolling_folds()
        
        # 2. 获取扩展股票数据
        raw_df = get_extended_data()
        
        # 3. 特征工程
        feature_df = engineer_features(raw_df)
        
        # 4. 生成目标变量
        full_df = generate_target_variables(feature_df)
        
        # 5. 导出滚动验证数据
        output_file = export_rolling_validation_data(full_df, rolling_folds)
        
        # 6. 输出统计信息
        print("\n" + "=" * 50)
        print("滚动验证数据收集完成!")
        print("=" * 50)
        print(f"数据时间范围: {START_DATE} 至 {END_DATE}")
        print(f"总特征数量: {full_df.shape[1]}")
        print(f"滚动验证窗口数: {len(rolling_folds)}")
        print(f"时间窗口参数: {get_all_time_windows()}")
        print(f"预测目标: {TARGET_HORIZONS}天收益率")
        print(f"数据清洗交易日数: {PURGE_DAYS}个交易日")
        print(f"输出文件: {output_file}")
        
        return output_file, rolling_folds
        
    except Exception as e:
        print(f"滚动验证数据收集失败: {e}")
        import traceback
        traceback.print_exc()
        return None, None

if __name__ == "__main__":
    main()

QQQ 滚动验证数据收集系统启动
Fold 1: Train(2009-01-01 to 2013-12-31) -> Test(2014-01-01 to 2014-12-31)
Fold 2: Train(2009-01-01 to 2014-12-31) -> Test(2015-01-01 to 2015-12-31)
Fold 3: Train(2009-01-01 to 2015-12-31) -> Test(2016-01-01 to 2016-12-31)
Fold 4: Train(2009-01-01 to 2016-12-31) -> Test(2017-01-01 to 2017-12-31)
Fold 5: Train(2009-01-01 to 2017-12-31) -> Test(2018-01-01 to 2018-12-31)
Fold 6: Train(2009-01-01 to 2018-12-31) -> Test(2019-01-01 to 2019-12-31)
Fold 7: Train(2009-01-01 to 2019-12-31) -> Test(2020-01-01 to 2020-12-31)
Fold 8: Train(2009-01-01 to 2020-12-31) -> Test(2021-01-01 to 2021-12-31)
Fold 9: Train(2009-01-01 to 2021-12-31) -> Test(2022-01-01 to 2022-12-31)
Fold 10: Train(2009-01-01 to 2022-12-31) -> Test(2023-01-01 to 2023-12-31)
Fold 11: Train(2009-01-01 to 2023-12-31) -> Test(2024-01-01 to 2024-12-31)
Fold 12: Train(2009-01-01 to 2024-12-31) -> Test(2025-01-01 to 2025-06-30)

总共生成 12 个滚动验证窗口
获取 QQQ 扩展数据...
获取到 4936 条原始数据记录
开始特征工程...
特征工程完成，共生成 118 个特征
生成目标变量...
目标变量

## 保存选中次数最多的特征和参数

In [5]:
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
import json
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

# ================================
# 配置参数
# ================================
DATA_PATH = 'QQQ_Rolling_Validation/QQQ_rolling_validation_dataset.xlsx'
OUTPUT_DIR = 'QQQ_XGBoost_IC_Analysis'
TARGET_NAME = '目标_未来10d回报率'

# XGBoost参数网格
XGBOOST_PARAM_GRID = {
    'n_estimators': [100, 200, 300, 400],      # 适中数量，结合early_stopping可调节
    'learning_rate': [0.01, 0.03, 0.05, 0.1], # 小步长提高稳定性
    'max_depth': [3, 4, 5, 6] ,                    # 控制复杂度，避免过拟合
    'subsample': [0.7, 0.8, 0.9],                  # 采样比例，防止过拟合
    'colsample_bytree': [0.7, 0.8, 0.9],           # 特征采样比例
    'reg_alpha': [0, 0.1, 0.5],                    # L1正则，适当防止过拟合
    'reg_lambda': [1, 1.5, 2],                     # L2正则，适当防止过拟合
    'min_child_weight': [1, 3, 5],                 # 控制子叶最小样本权重，防止过拟合
    'gamma': [0, 0.1, 0.2]                         # 节点分裂最小损失减少量，控制复杂度
}

# 策略参数
TOP_N_FEATURES = 30
RANDOM_SEARCH_ITER = 100
SHAP_SAMPLE_SIZE = 500
# MODIFIED: Changed from a top_n parameter to a minimum count threshold
MIN_SELECTION_COUNT = 8  # <--- 新增: 用于筛选稳定特征的最小选中次数

# ================================
# 核心函数
# ================================

def set_chinese_font():
    """设置中文字体"""
    try:
        plt.rcParams['font.sans-serif'] = ['SimHei']
        plt.rcParams['axes.unicode_minus'] = False
    except:
        pass

def get_available_folds(data_path):
    """获取所有可用的fold编号"""
    excel_file = pd.ExcelFile(data_path)
    fold_sheets = [sheet for sheet in excel_file.sheet_names if sheet.startswith('Fold') and sheet.endswith('_训练集')]
    fold_numbers = [int(sheet.split('Fold')[1].split('_')[0]) for sheet in fold_sheets]
    return sorted(fold_numbers)

def load_fold_data(data_path, fold_number, target_name):
    """加载fold数据"""
    train_sheet = f'Fold{fold_number:02d}_训练集'
    train_df = pd.read_excel(data_path, sheet_name=train_sheet, index_col=0, parse_dates=True)
    train_df = train_df.dropna()
    
    test_sheet = f'Fold{fold_number:02d}_测试集'
    test_df = pd.read_excel(data_path, sheet_name=test_sheet, index_col=0, parse_dates=True)
    test_df = test_df.dropna()
    
    # 识别特征列
    all_targets = [col for col in train_df.columns if str(col).startswith('目标_')]
    base_cols = ['开盘价', '最高价', '最低价', '收盘价', '成交量', '成交额', '日收益率']
    feature_cols = [col for col in train_df.columns if col not in all_targets + base_cols]
    
    X_train = train_df[feature_cols]
    y_train = train_df[target_name]
    X_test = test_df[feature_cols]
    y_test = test_df[target_name]
    
    return X_train, y_train, X_test, y_test

def calculate_ic(predictions, targets, method='spearman'):
    """计算IC值"""
    if method == 'pearson':
        ic, p_value = stats.pearsonr(predictions, targets)
    elif method == 'spearman':
        ic, p_value = stats.spearmanr(predictions, targets)
    return ic, p_value

class DynamicTimeSeriesSplit:
    """动态时间序列分割，根据fold自动调整分割数量"""
    
    def __init__(self, fold_number=1):
        self.fold_number = fold_number
    
    def split(self, X, y=None, groups=None):
        """
        根据fold动态调整分割策略
        fold1: 2009-2010/2011; 2009-2011/2012; 2009-2012/2013
        fold2: 2009-2010/2011; 2009-2011/2012; 2009-2012/2013; 2009-2013/2014
        以此类推...
        """
        # 获取所有年份
        years = sorted(X.index.year.unique())
        
        print(f"    动态TimeSeriesSplit调参过程 (Fold {self.fold_number}):")
        print(f"    可用年份: {years}")
        print(f"    总数据量: {len(X)} 个样本")
        
        splits = []
        
        # 确保有足够的年份进行交叉验证
        if len(years) < 3:
            print(f"    警告: 年份数量不足({len(years)}年)，无法进行时间序列交叉验证")
            return splits
        
        # 动态计算splits数量：fold1有3个split，fold2有4个split，以此类推
        # 但不能超过可用年份数量的限制
        target_splits = 2 + self.fold_number  # fold1->3, fold2->4, fold3->5...
        max_possible_splits = len(years) - 1   # 至少需要1年做验证集
        actual_splits = min(target_splits, max_possible_splits)
        
        print(f"    目标splits数: {target_splits}, 实际splits数: {actual_splits}")
        
        for i in range(actual_splits):
            # 训练集：从第一年到第(i+1)年
            train_years = years[:i+2]  # i=0时取前2年，i=1时取前3年...
            
            # 验证集：第(i+2)年的下一年
            val_year_index = i + 2
            if val_year_index < len(years):
                val_year = years[val_year_index]
            else:
                break
            
            # 获取训练集数据
            train_mask = X.index.year.isin(train_years)
            train_indices = X.index[train_mask]
            
            # 从训练集中去掉最后10个交易日
            if len(train_indices) > 10:
                train_indices = train_indices[:-10]
            
            # 获取验证集数据
            val_mask = X.index.year == val_year
            val_indices = X.index[val_mask]
            
            if len(train_indices) > 0 and len(val_indices) > 0:
                # 转换为位置索引
                train_pos = [X.index.get_loc(idx) for idx in train_indices]
                val_pos = [X.index.get_loc(idx) for idx in val_indices]
                
                print(f"    Split {i+1}:")
                print(f"      训练集: {train_years} (去掉最后10个交易日)")
                print(f"      训练集时间: {train_indices[0].strftime('%Y-%m-%d')} 至 {train_indices[-1].strftime('%Y-%m-%d')} ({len(train_indices)} 样本)")
                print(f"      验证集: {val_year}年")
                print(f"      验证集时间: {val_indices[0].strftime('%Y-%m-%d')} 至 {val_indices[-1].strftime('%Y-%m-%d')} ({len(val_indices)} 样本)")
                
                splits.append((train_pos, val_pos))
        
        print(f"    总共生成 {len(splits)} 个splits")
        return splits
    
    def get_n_splits(self, X=None, y=None, groups=None):
        if X is not None:
            years = sorted(X.index.year.unique())
            target_splits = 2 + self.fold_number
            max_possible_splits = len(years) - 1
            return min(target_splits, max_possible_splits)
        return 2 + self.fold_number

def ic_scorer(y_true, y_pred):
    """IC评分函数，用于交叉验证"""
    ic, _ = calculate_ic(y_pred, y_true, method='spearman')
    return ic

def hyperparameter_tuning(X_train, y_train, fold_number):
    """超参数调优"""
    print("  进行超参数调优...")
    
    base_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1)
    tscv = DynamicTimeSeriesSplit(fold_number=fold_number)
    
    # 创建IC评分器
    ic_scorer_func = make_scorer(ic_scorer, greater_is_better=True)
    
    random_search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=XGBOOST_PARAM_GRID,
        n_iter=RANDOM_SEARCH_ITER,
        cv=tscv,
        scoring=ic_scorer_func,  # 使用IC作为评分标准
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    
    random_search.fit(X_train, y_train)
    
    print(f"    最佳CV IC得分: {random_search.best_score_:.6f}")
    print(f"    使用的CV splits数: {tscv.get_n_splits(X_train)}")
    return random_search.best_params_

def feature_selection_shap(X_train, y_train, best_params, n_features=TOP_N_FEATURES):
    """基于SHAP的特征选择"""
    print(f"  进行特征选择(Top {n_features})...")
    
    model = xgb.XGBRegressor(**best_params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    
    # SHAP分析
    sample_size = min(SHAP_SAMPLE_SIZE, len(X_train))
    X_sample = X_train.sample(n=sample_size, random_state=42)
    
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    
    # 计算特征重要性
    importance = np.abs(shap_values).mean(0)
    importance_df = pd.DataFrame({
        'Feature': X_train.columns,
        'SHAP_Importance': importance
    }).sort_values('SHAP_Importance', ascending=False).reset_index(drop=True)
    
    top_features = importance_df.head(n_features)['Feature'].tolist()
    
    return top_features, importance_df

def train_final_model(X_train, y_train, X_test, y_test, best_params, top_features):
    """训练最终模型并计算IC"""
    print("  训练最终模型...")
    
    X_train_selected = X_train[top_features]
    X_test_selected = X_test[top_features]
    
    final_model = xgb.XGBRegressor(**best_params, random_state=42, n_jobs=-1)
    final_model.fit(X_train_selected, y_train)
    
    y_train_pred = final_model.predict(X_train_selected)
    y_test_pred = final_model.predict(X_test_selected)
    
    # 转换为Series并保持原索引
    y_train_pred = pd.Series(y_train_pred, index=y_train.index)
    y_test_pred = pd.Series(y_test_pred, index=y_test.index)
    
    # 计算IC (使用Spearman相关系数)
    train_ic, train_ic_pval = calculate_ic(y_train_pred.values, y_train.values, method='spearman')
    test_ic, test_ic_pval = calculate_ic(y_test_pred.values, y_test.values, method='spearman')
    
    print(f"    训练集Spearman IC: {train_ic:.4f} (p={train_ic_pval:.4f})")
    print(f"    测试集Spearman IC: {test_ic:.4f} (p={test_ic_pval:.4f})")
    
    return final_model, y_train_pred, y_test_pred, train_ic, test_ic, train_ic_pval, test_ic_pval

def create_prediction_distribution_plot(y_train_pred, y_test_pred, y_train, y_test, fold_number, fold_dir):
    """新增功能1: 为每个fold创建预测值分布图"""
    set_chinese_font()
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 训练集预测值分布
    axes[0, 0].hist(y_train_pred, bins=30, alpha=0.7, color='blue', edgecolor='black', label='预测值')
    axes[0, 0].axvline(y_train_pred.mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {y_train_pred.mean():.4f}')
    axes[0, 0].set_title(f'Fold {fold_number:02d} - 训练集预测值分布')
    axes[0, 0].set_xlabel('预测值')
    axes[0, 0].set_ylabel('频数')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 测试集预测值分布
    axes[0, 1].hist(y_test_pred, bins=30, alpha=0.7, color='green', edgecolor='black', label='预测值')
    axes[0, 1].axvline(y_test_pred.mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {y_test_pred.mean():.4f}')
    axes[0, 1].set_title(f'Fold {fold_number:02d} - 测试集预测值分布')
    axes[0, 1].set_xlabel('预测值')
    axes[0, 1].set_ylabel('频数')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 训练集：预测值vs真实值对比分布
    axes[1, 0].hist(y_train, bins=30, alpha=0.5, color='blue', edgecolor='black', label='真实值')
    axes[1, 0].hist(y_train_pred, bins=30, alpha=0.5, color='red', edgecolor='black', label='预测值')
    axes[1, 0].set_title(f'Fold {fold_number:02d} - 训练集预测vs真实分布')
    axes[1, 0].set_xlabel('值')
    axes[1, 0].set_ylabel('频数')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 测试集：预测值vs真实值对比分布
    axes[1, 1].hist(y_test, bins=30, alpha=0.5, color='blue', edgecolor='black', label='真实值')
    axes[1, 1].hist(y_test_pred, bins=30, alpha=0.5, color='red', edgecolor='black', label='预测值')
    axes[1, 1].set_title(f'Fold {fold_number:02d} - 测试集预测vs真实分布')
    axes[1, 1].set_xlabel('值')
    axes[1, 1].set_ylabel('频数')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # 添加统计信息
    fig.suptitle(f'Fold {fold_number:02d} 预测值分布分析', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(fold_dir, f'prediction_distribution.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  已生成Fold {fold_number:02d}的预测值分布图")

def process_single_fold(data_path, fold_number, target_name, output_dir):
    """处理单个fold"""
    print(f"\n处理 Fold {fold_number}")
    print("=" * 50)
    
    # 加载数据
    X_train, y_train, X_test, y_test = load_fold_data(data_path, fold_number, target_name)
    
    print(f"数据规模: 训练集 {X_train.shape}, 测试集 {X_test.shape}")
    print(f"训练集时间范围: {X_train.index[0].strftime('%Y-%m-%d')} 至 {X_train.index[-1].strftime('%Y-%m-%d')}")
    print(f"测试集时间范围: {X_test.index[0].strftime('%Y-%m-%d')} 至 {X_test.index[-1].strftime('%Y-%m-%d')}")
    
    # 超参数调优（传入fold_number）
    best_params = hyperparameter_tuning(X_train, y_train, fold_number)
    
    # 特征选择
    top_features, importance_df = feature_selection_shap(X_train, y_train, best_params)
    
    # 训练最终模型
    final_model, y_train_pred, y_test_pred, train_ic, test_ic, train_ic_pval, test_ic_pval = train_final_model(
        X_train, y_train, X_test, y_test, best_params, top_features)
    
    # 输出最优超参数
    print(f"  最优超参数:")
    for param, value in best_params.items():
        print(f"    {param}: {value}")
    
    # 保存结果
    fold_dir = os.path.join(output_dir, f'Fold{fold_number:02d}')
    os.makedirs(fold_dir, exist_ok=True)
    
    # 保存预测结果
    results_df = pd.DataFrame({
        'Date': y_train.index.tolist() + y_test.index.tolist(),
        'Type': ['Train'] * len(y_train) + ['Test'] * len(y_test),
        'Prediction': y_train_pred.tolist() + y_test_pred.tolist(),
        'Actual': y_train.tolist() + y_test.tolist()
    })
    results_df.to_csv(os.path.join(fold_dir, 'predictions.csv'), index=False)
    
    # 保存重要特征
    importance_df.to_csv(os.path.join(fold_dir, 'feature_importance.csv'), index=False)
    
    # 新增功能1: 创建预测值分布图
    create_prediction_distribution_plot(y_train_pred, y_test_pred, y_train, y_test, fold_number, fold_dir)
    
    return {
        'fold': fold_number,
        'train_ic': train_ic,
        'test_ic': test_ic,
        'train_ic_pval': train_ic_pval,
        'test_ic_pval': test_ic_pval,
        'top_features': top_features,
        'best_params': best_params,
        'test_predictions': y_test_pred,
        'test_actual': y_test,
        'train_predictions': y_train_pred,
        'train_actual': y_train
    }

def export_combined_test_predictions(all_results, output_dir):
    """新增功能2: 导出所有fold测试集拼起来的预测值和真实值到Excel"""
    print(f"\n导出合并测试集预测结果到Excel")
    print("=" * 50)
    
    # 收集所有测试集数据
    combined_data = []
    
    for result in all_results:
        fold_number = result['fold']
        test_predictions = result['test_predictions']
        test_actual = result['test_actual']
        
        # 创建每个fold的数据框
        fold_df = pd.DataFrame({
            'Date': test_predictions.index,
            'Fold': fold_number,
            'Prediction': test_predictions.values,
            'Actual': test_actual.values
        })
        
        combined_data.append(fold_df)
    
    # 合并所有数据
    combined_test_df = pd.concat(combined_data, ignore_index=True)
    
    # 按日期排序
    combined_test_df = combined_test_df.sort_values('Date').reset_index(drop=True)
    
    # 计算误差和其他统计量
    combined_test_df['Error'] = combined_test_df['Prediction'] - combined_test_df['Actual']
    combined_test_df['Abs_Error'] = np.abs(combined_test_df['Error'])
    combined_test_df['Squared_Error'] = combined_test_df['Error'] ** 2
    
    # 计算整体统计
    overall_ic, overall_pval = calculate_ic(
        combined_test_df['Prediction'].values, 
        combined_test_df['Actual'].values, 
        method='spearman'
    )
    
    # 创建统计摘要
    summary_stats = pd.DataFrame({
        'Metric': ['Sample Count', 'Overall IC', 'IC P-value', 'Mean Prediction', 'Mean Actual', 
                   'Std Prediction', 'Std Actual', 'MAE', 'MSE', 'RMSE'],
        'Value': [
            len(combined_test_df),
            overall_ic,
            overall_pval,
            combined_test_df['Prediction'].mean(),
            combined_test_df['Actual'].mean(),
            combined_test_df['Prediction'].std(),
            combined_test_df['Actual'].std(),
            combined_test_df['Abs_Error'].mean(),
            combined_test_df['Squared_Error'].mean(),
            np.sqrt(combined_test_df['Squared_Error'].mean())
        ]
    })
    
    # 保存到Excel
    excel_path = os.path.join(output_dir, 'combined_test_predictions.xlsx')
    
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # 主数据表
        combined_test_df.to_excel(writer, sheet_name='测试集预测数据', index=False)
        
        # 统计摘要表
        summary_stats.to_excel(writer, sheet_name='统计摘要', index=False)
        
        # 按Fold的汇总统计
        fold_summary = combined_test_df.groupby('Fold').agg({
            'Prediction': ['mean', 'std', 'min', 'max'],
            'Actual': ['mean', 'std', 'min', 'max'],
            'Error': ['mean', 'std'],
            'Abs_Error': 'mean'
        }).round(6)
        fold_summary.to_excel(writer, sheet_name='各Fold统计')
    
    print(f"  已导出合并测试集数据到: {excel_path}")
    print(f"  总样本数: {len(combined_test_df)}")
    print(f"  时间范围: {combined_test_df['Date'].min().strftime('%Y-%m-%d')} 至 {combined_test_df['Date'].max().strftime('%Y-%m-%d')}")
    print(f"  整体测试集IC: {overall_ic:.4f} (p={overall_pval:.4f})")
    
    return combined_test_df, summary_stats

def create_summary_analysis(all_results, output_dir):
    """创建汇总分析"""
    print(f"\n创建汇总分析")
    print("=" * 50)
    
    # 收集结果
    summary_data = []
    for result in all_results:
        summary_data.append({
            'Fold': result['fold'],
            'Train_IC': result['train_ic'],
            'Test_IC': result['test_ic'],
            'Train_IC_PValue': result['train_ic_pval'],
            'Test_IC_PValue': result['test_ic_pval'],
            'IC_Significant': result['test_ic_pval'] < 0.05
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    # 计算统计指标
    print(f"各Fold IC结果:")
    for _, row in summary_df.iterrows():
        significance = "***" if row['Test_IC_PValue'] < 0.01 else "**" if row['Test_IC_PValue'] < 0.05 else "*" if row['Test_IC_PValue'] < 0.1 else ""
        print(f"  Fold {int(row['Fold']):02d}: 训练IC={row['Train_IC']:.4f}, 测试IC={row['Test_IC']:.4f}{significance} (p={row['Test_IC_PValue']:.4f})")
    
    print(f"\n整体IC统计:")
    print(f"平均IC (训练集): {summary_df['Train_IC'].mean():.4f} ± {summary_df['Train_IC'].std():.4f}")
    print(f"平均IC (测试集): {summary_df['Test_IC'].mean():.4f} ± {summary_df['Test_IC'].std():.4f}")
    print(f"IC显著性(p<0.05): {summary_df['IC_Significant'].sum()}/{len(summary_df)} ({summary_df['IC_Significant'].mean():.1%})")
    print(f"IC胜率: {(summary_df['Test_IC'] > 0).sum()}/{len(summary_df)} ({(summary_df['Test_IC'] > 0).mean():.1%})")
    
    # 特征稳定性分析
    all_features = {}
    for result in all_results:
        for feature in result['top_features']:
            all_features[feature] = all_features.get(feature, 0) + 1
    
    feature_stability = pd.DataFrame([
        {'Feature': feature, 'Count': count, 'Rate': count/len(all_results)}
        for feature, count in all_features.items()
    ]).sort_values('Count', ascending=False)
    
    # 创建全数据集IC分析
    overall_results = create_overall_ic_analysis(all_results, output_dir)
    
    # 新增功能2: 导出合并测试集预测结果
    combined_test_df, summary_stats = export_combined_test_predictions(all_results, output_dir)
    
    # 保存结果
    with pd.ExcelWriter(os.path.join(output_dir, 'ic_analysis_results.xlsx')) as writer:
        summary_df.to_excel(writer, sheet_name='各Fold_IC结果', index=False)
        feature_stability.to_excel(writer, sheet_name='特征稳定性', index=False)
    
    # 创建可视化
    create_ic_visualization(summary_df, feature_stability, output_dir)
    
    return summary_df, feature_stability

def create_overall_ic_analysis(all_results, output_dir):
    """创建全数据集的IC分析"""
    print(f"\n创建全数据集IC分析")
    print("=" * 50)
    
    # 拼接所有测试集的预测和实际值
    all_test_predictions = []
    all_test_actual = []
    all_train_predictions = []
    all_train_actual = []
    all_dates_test = []
    all_dates_train = []
    
    for result in all_results:
        all_test_predictions.extend(result['test_predictions'].tolist())
        all_test_actual.extend(result['test_actual'].tolist())
        all_train_predictions.extend(result['train_predictions'].tolist())
        all_train_actual.extend(result['train_actual'].tolist())
        all_dates_test.extend(result['test_predictions'].index.tolist())
        all_dates_train.extend(result['train_predictions'].index.tolist())
    
    # 计算全数据集IC
    overall_test_ic, overall_test_ic_pval = calculate_ic(all_test_predictions, all_test_actual, method='spearman')
    overall_train_ic, overall_train_ic_pval = calculate_ic(all_train_predictions, all_train_actual, method='spearman')
    
    # 输出结果
    print(f"全数据集IC分析:")
    print(f"训练集合并IC: {overall_train_ic:.4f} (p={overall_train_ic_pval:.4f})")
    print(f"测试集合并IC: {overall_test_ic:.4f} (p={overall_test_ic_pval:.4f})")
    print(f"测试集数据点数: {len(all_test_predictions)}")
    print(f"训练集数据点数: {len(all_train_predictions)}")
    
    if all_dates_test:
        print(f"测试集时间跨度: {min(all_dates_test).strftime('%Y-%m-%d')} 至 {max(all_dates_test).strftime('%Y-%m-%d')}")
    if all_dates_train:
        print(f"训练集时间跨度: {min(all_dates_train).strftime('%Y-%m-%d')} 至 {max(all_dates_train).strftime('%Y-%m-%d')}")
    
    # 保存全数据集结果
    overall_summary = pd.DataFrame({
        'Dataset': ['训练集合并', '测试集合并'],
        'IC': [overall_train_ic, overall_test_ic],
        'P_Value': [overall_train_ic_pval, overall_test_ic_pval],
        'Significant': [overall_train_ic_pval < 0.05, overall_test_ic_pval < 0.05],
        'Data_Points': [len(all_train_predictions), len(all_test_predictions)]
    })
    
    # 创建散点图
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            f'训练集预测vs实际 (IC: {overall_train_ic:.4f})',
            f'测试集预测vs实际 (IC: {overall_test_ic:.4f})'
        ]
    )
    
    # 训练集散点图
    fig.add_trace(go.Scatter(
        x=all_train_actual,
        y=all_train_predictions,
        mode='markers',
        name='训练集',
        marker=dict(color='blue', opacity=0.6, size=3),
        showlegend=False
    ), row=1, col=1)
    
    # 测试集散点图
    fig.add_trace(go.Scatter(
        x=all_test_actual,
        y=all_test_predictions,
        mode='markers',
        name='测试集',
        marker=dict(color='red', opacity=0.6, size=3),
        showlegend=False
    ), row=1, col=2)
    
    fig.update_layout(
        title='全数据集预测效果分析',
        height=500
    )
    
    fig.update_xaxes(title_text="实际值")
    fig.update_yaxes(title_text="预测值")
    
    # 保存结果
    with pd.ExcelWriter(os.path.join(output_dir, 'overall_ic_analysis.xlsx')) as writer:
        overall_summary.to_excel(writer, sheet_name='全数据集IC分析', index=False)
        
        # 保存详细数据
        overall_data = pd.DataFrame({
            'Date': all_dates_train + all_dates_test,
            'Type': ['Train'] * len(all_dates_train) + ['Test'] * len(all_dates_test),
            'Prediction': all_train_predictions + all_test_predictions,
            'Actual': all_train_actual + all_test_actual
        })
        overall_data.to_excel(writer, sheet_name='全数据集详细数据', index=False)
    
    fig.write_html(os.path.join(output_dir, 'overall_ic_scatter.html'))
    
    return {
        'overall_train_ic': overall_train_ic,
        'overall_test_ic': overall_test_ic,
        'summary': overall_summary
    }

def create_ic_visualization(summary_df, feature_stability, output_dir):
    """创建IC可视化"""
    set_chinese_font()
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # IC分布直方图
    axes[0, 0].hist(summary_df['Test_IC'], bins=10, alpha=0.7, color='blue', edgecolor='black')
    axes[0, 0].axvline(summary_df['Test_IC'].mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {summary_df["Test_IC"].mean():.3f}')
    axes[0, 0].axvline(0, color='black', linestyle='-', alpha=0.5)
    axes[0, 0].set_title('测试集IC分布')
    axes[0, 0].set_xlabel('IC值')
    axes[0, 0].set_ylabel('频数')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 各Fold IC对比
    x_pos = range(len(summary_df))
    colors = ['green' if ic > 0 else 'red' for ic in summary_df['Test_IC']]
    bars = axes[0, 1].bar(x_pos, summary_df['Test_IC'], color=colors, alpha=0.7)
    axes[0, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
    axes[0, 1].axhline(y=summary_df['Test_IC'].mean(), color='blue', linestyle='--', linewidth=2, label=f'平均IC: {summary_df["Test_IC"].mean():.3f}')
    axes[0, 1].set_title('各Fold测试集IC')
    axes[0, 1].set_xlabel('Fold')
    axes[0, 1].set_ylabel('IC值')
    axes[0, 1].set_xticks(x_pos)
    axes[0, 1].set_xticklabels([f'F{int(fold):02d}' for fold in summary_df['Fold']])
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 添加显著性标记
    for i, (bar, pval) in enumerate(zip(bars, summary_df['Test_IC_PValue'])):
        height = bar.get_height()
        if pval < 0.01:
            axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + (0.01 if height >= 0 else -0.02), 
                          '***', ha='center', va='bottom' if height >= 0 else 'top', fontsize=12, fontweight='bold')
        elif pval < 0.05:
            axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + (0.01 if height >= 0 else -0.02), 
                          '**', ha='center', va='bottom' if height >= 0 else 'top', fontsize=12, fontweight='bold')
        elif pval < 0.1:
            axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + (0.01 if height >= 0 else -0.02), 
                          '*', ha='center', va='bottom' if height >= 0 else 'top', fontsize=12, fontweight='bold')
    
    # 训练vs测试IC对比
    axes[1, 0].scatter(summary_df['Train_IC'], summary_df['Test_IC'], alpha=0.7, s=60, color='blue')
    axes[1, 0].plot([-0.3, 0.3], [-0.3, 0.3], 'r--', alpha=0.5, label='y=x')
    axes[1, 0].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    axes[1, 0].axvline(x=0, color='black', linestyle='-', alpha=0.3)
    axes[1, 0].set_xlabel('训练集IC')
    axes[1, 0].set_ylabel('测试集IC')
    axes[1, 0].set_title('训练集vs测试集IC')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 特征稳定性
    top_features = feature_stability.head(15)
    y_pos = range(len(top_features))
    axes[1, 1].barh(y_pos, top_features['Count'], alpha=0.7, color='green')
    axes[1, 1].set_yticks(y_pos)
    axes[1, 1].set_yticklabels(top_features['Feature'], fontsize=8)
    axes[1, 1].set_xlabel('选中次数')
    axes[1, 1].set_title('Top 15 稳定特征')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'ic_analysis_summary.png'), dpi=300, bbox_inches='tight')
    plt.close()

# ================================
# 新增保存功能
# ================================

# MODIFIED: Changed function to accept min_count instead of top_n
def save_stable_features_and_hyperparams(all_results, output_dir, min_count=5):
    """
    保存稳定特征和超参数
    Args:
        all_results (list): 所有fold的结果列表
        output_dir (str): 输出目录
        min_count (int): 特征被选中的最小次数，用于筛选稳定特征
    """
    print(f"\n保存稳定特征和超参数")
    print("=" * 50)
    
    # 1. 提取并保存稳定特征
    all_features = {}
    for result in all_results:
        for feature in result['top_features']:
            all_features[feature] = all_features.get(feature, 0) + 1
    
    feature_stability = []
    for feature, count in all_features.items():
        feature_stability.append({
            'Feature': feature,
            'Count': count,
            'Rate': count / len(all_results)
        })
    
    feature_stability.sort(key=lambda x: x['Count'], reverse=True)
    
    # MODIFIED: Filter features by min_count instead of taking the top_n
    stable_feature_names = [f['Feature'] for f in feature_stability if f['Count'] >= min_count]
    
    # 保存稳定特征
    stable_features_file = os.path.join(output_dir, 'stable_features.txt')
    with open(stable_features_file, 'w', encoding='utf-8') as f:
        for feature in stable_feature_names:
            f.write(feature + '\n')
    
    # MODIFIED: Updated print statement to reflect the new logic
    print(f"已保存选择次数 >= {min_count} 的稳定特征 ({len(stable_feature_names)}个) 到: {stable_features_file}")
    
    # 2. 保存每个fold的超参数
    hyperparams_data = []
    for result in all_results:
        fold_params = {'Fold': result['fold']}
        fold_params.update(result['best_params'])
        hyperparams_data.append(fold_params)
    
    # 保存为JSON
    hyperparams_json = os.path.join(output_dir, 'all_fold_hyperparams.json')
    with open(hyperparams_json, 'w', encoding='utf-8') as f:
        json.dump(hyperparams_data, f, indent=2, ensure_ascii=False)
    
    print(f"已保存各fold超参数到: {hyperparams_json}")
    
    return stable_feature_names, hyperparams_data

def main():
    """主函数"""
    if not os.path.exists(DATA_PATH):
        print(f"错误：数据文件不存在 {DATA_PATH}")
        return
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    available_folds = get_available_folds(DATA_PATH)
    
    print(f"XGBoost IC分析 - 动态时间序列分割")
    print(f"发现 {len(available_folds)} 个fold")
    print(f"目标变量: {TARGET_NAME}")
    print(f"Top特征数 (单次选择): {TOP_N_FEATURES}") # Renamed for clarity
    print(f"稳定特征筛选阈值: 选中次数 >= {MIN_SELECTION_COUNT}") # Added for clarity
    print(f"IC计算方法: Spearman相关系数")
    print(f"分割策略: 动态调整 - Fold1用3个splits，Fold2用4个splits，以此类推")
    
    all_results = []
    for fold_number in available_folds:
        try:
            result = process_single_fold(DATA_PATH, fold_number, TARGET_NAME, OUTPUT_DIR)
            all_results.append(result)
        except Exception as e:
            print(f"Fold {fold_number} 处理失败: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    if not all_results:
        print("错误：没有成功处理任何fold")
        return
    
    # 创建汇总分析
    summary_df, feature_stability = create_summary_analysis(all_results, OUTPUT_DIR)
    
    # MODIFIED: Call the function with the new min_count parameter
    stable_features, best_hyperparams = save_stable_features_and_hyperparams(all_results, OUTPUT_DIR, min_count=MIN_SELECTION_COUNT)
    
    print(f"\n分析完成！")
    print(f"成功处理 {len(all_results)} 个fold")
    print(f"结果保存至: {OUTPUT_DIR}")
    
    # 最终总结
    print(f"\n" + "="*60)
    print(f"最终IC总结:")
    print(f"="*60)
    print(f"各Fold平均测试IC: {summary_df['Test_IC'].mean():.4f} ± {summary_df['Test_IC'].std():.4f}")
    print(f"IC胜率: {(summary_df['Test_IC'] > 0).mean():.1%}")
    print(f"显著IC(p<0.05)占比: {summary_df['IC_Significant'].mean():.1%}")
    print(f"\n已保存:")
    print(f"- 稳定特征 (选中次数 >= {MIN_SELECTION_COUNT}): {len(stable_features)} 个")
    print(f"- 最优超参数配置")
    print(f"- 完整分析结果")

if __name__ == "__main__":
    main()

XGBoost IC分析 - 动态时间序列分割
发现 12 个fold
目标变量: 目标_未来10d回报率
Top特征数 (单次选择): 30
稳定特征筛选阈值: 选中次数 >= 8
IC计算方法: Spearman相关系数
分割策略: 动态调整 - Fold1用3个splits，Fold2用4个splits，以此类推

处理 Fold 1
数据规模: 训练集 (1247, 111), 测试集 (252, 111)
训练集时间范围: 2009-01-02 至 2013-12-16
测试集时间范围: 2014-01-02 至 2014-12-31
  进行超参数调优...
    动态TimeSeriesSplit调参过程 (Fold 1):
    可用年份: [2009, 2010, 2011, 2012, 2013]
    总数据量: 1247 个样本
    目标splits数: 3, 实际splits数: 3
    Split 1:
      训练集: [2009, 2010] (去掉最后10个交易日)
      训练集时间: 2009-01-02 至 2010-12-16 (494 样本)
      验证集: 2011年
      验证集时间: 2011-01-03 至 2011-12-30 (252 样本)
    Split 2:
      训练集: [2009, 2010, 2011] (去掉最后10个交易日)
      训练集时间: 2009-01-02 至 2011-12-15 (746 样本)
      验证集: 2012年
      验证集时间: 2012-01-03 至 2012-12-31 (249 样本)
    Split 3:
      训练集: [2009, 2010, 2011, 2012] (去掉最后10个交易日)
      训练集时间: 2009-01-02 至 2012-12-14 (995 样本)
      验证集: 2013年
      验证集时间: 2013-01-02 至 2013-12-16 (242 样本)
    总共生成 3 个splits
    最佳CV IC得分: 0.088876
    使用的CV splits数: 3
  进行特征选择(Top 30)...
  训练最

## 利用保存的参数和特征训练模型

In [17]:
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

# ================================
# 配置参数
# ================================
DATA_PATH = 'QQQ_Rolling_Validation/QQQ_rolling_validation_dataset.xlsx'
OUTPUT_DIR = 'QQQ_XGBoost_Stable_Features_Analysis(Regression)'
TARGET_NAME = '目标_未来10d回报率'

# 稳定特征和超参数文件路径
STABLE_FEATURES_FILE = 'QQQ_XGBoost_IC_Analysis/stable_features.txt'
HYPERPARAMS_FILE = 'QQQ_XGBoost_IC_Analysis/all_fold_hyperparams.json'

# ================================
# 核心函数
# ================================

def set_chinese_font():
    """设置中文字体"""
    try:
        plt.rcParams['font.sans-serif'] = ['SimHei']
        plt.rcParams['axes.unicode_minus'] = False
    except:
        pass

def load_stable_features(file_path=STABLE_FEATURES_FILE):
    """加载稳定特征列表"""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"稳定特征文件不存在: {file_path}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        features = [line.strip() for line in f if line.strip()]
    
    print(f"从 {file_path} 加载了 {len(features)} 个稳定特征")
    return features

def load_hyperparams(file_path=HYPERPARAMS_FILE):
    """加载各fold的超参数"""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"超参数文件不存在: {file_path}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        hyperparams_data = json.load(f)
    
    # 转换为字典，以fold为key
    fold_hyperparams = {}
    for params in hyperparams_data:
        fold_num = params['Fold']
        fold_params = {k: v for k, v in params.items() if k != 'Fold'}
        fold_hyperparams[fold_num] = fold_params
    
    print(f"从 {file_path} 加载了 {len(fold_hyperparams)} 个fold的超参数")
    return fold_hyperparams

def get_available_folds(data_path):
    """获取所有可用的fold编号"""
    excel_file = pd.ExcelFile(data_path)
    fold_sheets = [sheet for sheet in excel_file.sheet_names if sheet.startswith('Fold') and sheet.endswith('_训练集')]
    fold_numbers = [int(sheet.split('Fold')[1].split('_')[0]) for sheet in fold_sheets]
    return sorted(fold_numbers)

def load_fold_data(data_path, fold_number, target_name, stable_features):
    """加载fold数据，只使用稳定特征"""
    train_sheet = f'Fold{fold_number:02d}_训练集'
    train_df = pd.read_excel(data_path, sheet_name=train_sheet, index_col=0, parse_dates=True)
    train_df = train_df.dropna()
    
    test_sheet = f'Fold{fold_number:02d}_测试集'
    test_df = pd.read_excel(data_path, sheet_name=test_sheet, index_col=0, parse_dates=True)
    test_df = test_df.dropna()
    
    # 检查稳定特征是否存在于数据中
    available_features = [f for f in stable_features if f in train_df.columns]
    missing_features = [f for f in stable_features if f not in train_df.columns]
    
    if missing_features:
        print(f"  警告: {len(missing_features)} 个稳定特征在数据中不存在")
        if len(missing_features) <= 5:
            print(f"  缺失特征: {missing_features}")
    
    print(f"  使用 {len(available_features)} 个可用稳定特征")
    
    # 使用可用的稳定特征
    X_train = train_df[available_features]
    y_train = train_df[target_name]
    X_test = test_df[available_features]
    y_test = test_df[target_name]
    
    return X_train, y_train, X_test, y_test, available_features

def calculate_ic(predictions, targets, method='spearman'):
    """计算IC值"""
    if method == 'pearson':
        ic, p_value = stats.pearsonr(predictions, targets)
    elif method == 'spearman':
        ic, p_value = stats.spearmanr(predictions, targets)
    return ic, p_value

def train_final_model(X_train, y_train, X_test, y_test, hyperparams):
    """训练最终模型并计算IC"""
    print("  训练最终模型...")
    
    # 添加固定参数
    final_params = hyperparams.copy()
    final_params.update({
        'objective': 'reg:squarederror',
        'random_state': 42,
        'n_jobs': -1
    })
    
    final_model = xgb.XGBRegressor(**final_params)
    final_model.fit(X_train, y_train)
    
    y_train_pred = final_model.predict(X_train)
    y_test_pred = final_model.predict(X_test)
    
    # 转换为Series并保持原索引
    y_train_pred = pd.Series(y_train_pred, index=y_train.index)
    y_test_pred = pd.Series(y_test_pred, index=y_test.index)
    
    # 计算IC (使用Spearman相关系数)
    train_ic, train_ic_pval = calculate_ic(y_train_pred.values, y_train.values, method='spearman')
    test_ic, test_ic_pval = calculate_ic(y_test_pred.values, y_test.values, method='spearman')
    
    print(f"    训练集Spearman IC: {train_ic:.4f} (p={train_ic_pval:.4f})")
    print(f"    测试集Spearman IC: {test_ic:.4f} (p={test_ic_pval:.4f})")
    
    return final_model, y_train_pred, y_test_pred, train_ic, test_ic, train_ic_pval, test_ic_pval

def create_prediction_distribution_plot(y_train_pred, y_test_pred, y_train, y_test, fold_number, fold_dir):
    """为每个fold创建预测值分布图"""
    set_chinese_font()
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 训练集预测值分布
    axes[0, 0].hist(y_train_pred, bins=30, alpha=0.7, color='blue', edgecolor='black', label='预测值')
    axes[0, 0].axvline(y_train_pred.mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {y_train_pred.mean():.4f}')
    axes[0, 0].set_title(f'Fold {fold_number:02d} - 训练集预测值分布')
    axes[0, 0].set_xlabel('预测值')
    axes[0, 0].set_ylabel('频数')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 测试集预测值分布
    axes[0, 1].hist(y_test_pred, bins=30, alpha=0.7, color='green', edgecolor='black', label='预测值')
    axes[0, 1].axvline(y_test_pred.mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {y_test_pred.mean():.4f}')
    axes[0, 1].set_title(f'Fold {fold_number:02d} - 测试集预测值分布')
    axes[0, 1].set_xlabel('预测值')
    axes[0, 1].set_ylabel('频数')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 训练集：预测值vs真实值对比分布
    axes[1, 0].hist(y_train, bins=30, alpha=0.5, color='blue', edgecolor='black', label='真实值')
    axes[1, 0].hist(y_train_pred, bins=30, alpha=0.5, color='red', edgecolor='black', label='预测值')
    axes[1, 0].set_title(f'Fold {fold_number:02d} - 训练集预测vs真实分布')
    axes[1, 0].set_xlabel('值')
    axes[1, 0].set_ylabel('频数')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 测试集：预测值vs真实值对比分布
    axes[1, 1].hist(y_test, bins=30, alpha=0.5, color='blue', edgecolor='black', label='真实值')
    axes[1, 1].hist(y_test_pred, bins=30, alpha=0.5, color='red', edgecolor='black', label='预测值')
    axes[1, 1].set_title(f'Fold {fold_number:02d} - 测试集预测vs真实分布')
    axes[1, 1].set_xlabel('值')
    axes[1, 1].set_ylabel('频数')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    fig.suptitle(f'Fold {fold_number:02d} 预测值分布分析', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(fold_dir, f'prediction_distribution.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  已生成Fold {fold_number:02d}的预测值分布图")

def create_fold_scatter_plot(y_train_pred, y_test_pred, y_train, y_test, fold_number, fold_dir):
    """为每个fold创建散点图"""
    # 计算IC
    train_ic, train_ic_pval = calculate_ic(y_train_pred.values, y_train.values, method='spearman')
    test_ic, test_ic_pval = calculate_ic(y_test_pred.values, y_test.values, method='spearman')
    
    # 创建散点图
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            f'训练集预测vs实际 (IC: {train_ic:.4f})',
            f'测试集预测vs实际 (IC: {test_ic:.4f})'
        ]
    )
    
    # 训练集散点图
    fig.add_trace(go.Scatter(
        x=y_train.values,
        y=y_train_pred.values,
        mode='markers',
        name='训练集',
        marker=dict(color='blue', opacity=0.6, size=3),
        showlegend=False
    ), row=1, col=1)
    
    # 测试集散点图
    fig.add_trace(go.Scatter(
        x=y_test.values,
        y=y_test_pred.values,
        mode='markers',
        name='测试集',
        marker=dict(color='red', opacity=0.6, size=3),
        showlegend=False
    ), row=1, col=2)
    
    fig.update_layout(
        title=f'Fold {fold_number:02d} 预测效果分析',
        height=500
    )
    
    fig.update_xaxes(title_text="实际值")
    fig.update_yaxes(title_text="预测值")
    
    # 保存图片
    scatter_file = os.path.join(fold_dir, f'fold_{fold_number:02d}_scatter.html')
    fig.write_html(scatter_file)
    
    print(f"  已生成Fold {fold_number:02d}的散点图: {scatter_file}")
    
    return train_ic, test_ic

def process_single_fold(data_path, fold_number, target_name, stable_features, fold_hyperparams, output_dir):
    """处理单个fold"""
    print(f"\n处理 Fold {fold_number}")
    print("=" * 50)
    
    # 获取该fold的超参数
    if fold_number not in fold_hyperparams:
        print(f"  错误: 没有找到Fold {fold_number}的超参数")
        return None
    
    hyperparams = fold_hyperparams[fold_number]
    print(f"  使用保存的超参数:")
    for param, value in hyperparams.items():
        print(f"    {param}: {value}")
    
    # 加载数据，使用稳定特征
    X_train, y_train, X_test, y_test, used_features = load_fold_data(data_path, fold_number, target_name, stable_features)
    
    print(f"数据规模: 训练集 {X_train.shape}, 测试集 {X_test.shape}")
    print(f"训练集时间范围: {X_train.index[0].strftime('%Y-%m-%d')} 至 {X_train.index[-1].strftime('%Y-%m-%d')}")
    print(f"测试集时间范围: {X_test.index[0].strftime('%Y-%m-%d')} 至 {X_test.index[-1].strftime('%Y-%m-%d')}")
    
    # 训练最终模型（跳过超参数调优和SHAP特征选择）
    final_model, y_train_pred, y_test_pred, train_ic, test_ic, train_ic_pval, test_ic_pval = train_final_model(
        X_train, y_train, X_test, y_test, hyperparams)
    
    # 保存结果
    fold_dir = os.path.join(output_dir, f'Fold{fold_number:02d}')
    os.makedirs(fold_dir, exist_ok=True)
    
    # 保存预测结果
    results_df = pd.DataFrame({
        'Date': y_train.index.tolist() + y_test.index.tolist(),
        'Type': ['Train'] * len(y_train) + ['Test'] * len(y_test),
        'Prediction': y_train_pred.tolist() + y_test_pred.tolist(),
        'Actual': y_train.tolist() + y_test.tolist()
    })
    results_df.to_csv(os.path.join(fold_dir, 'predictions.csv'), index=False)
    
    # 保存使用的特征列表
    pd.DataFrame({'Feature': used_features}).to_csv(os.path.join(fold_dir, 'used_features.csv'), index=False)
    
    # 保存使用的超参数
    with open(os.path.join(fold_dir, 'used_hyperparams.json'), 'w', encoding='utf-8') as f:
        json.dump(hyperparams, f, indent=2, ensure_ascii=False)
    
    # 创建预测值分布图
    create_prediction_distribution_plot(y_train_pred, y_test_pred, y_train, y_test, fold_number, fold_dir)
    
    # 创建散点图
    create_fold_scatter_plot(y_train_pred, y_test_pred, y_train, y_test, fold_number, fold_dir)
    
    return {
        'fold': fold_number,
        'train_ic': train_ic,
        'test_ic': test_ic,
        'train_ic_pval': train_ic_pval,
        'test_ic_pval': test_ic_pval,
        'used_features': used_features,
        'hyperparams': hyperparams,
        'test_predictions': y_test_pred,
        'test_actual': y_test,
        'train_predictions': y_train_pred,
        'train_actual': y_train
    }

def export_combined_test_predictions(all_results, output_dir):
    """导出所有fold测试集拼起来的预测值和真实值到Excel"""
    print(f"\n导出合并测试集预测结果到Excel")
    print("=" * 50)
    
    combined_data = []
    
    for result in all_results:
        fold_number = result['fold']
        test_predictions = result['test_predictions']
        test_actual = result['test_actual']
        
        fold_df = pd.DataFrame({
            'Date': test_predictions.index,
            'Fold': fold_number,
            'Prediction': test_predictions.values,
            'Actual': test_actual.values
        })
        
        combined_data.append(fold_df)
    
    combined_test_df = pd.concat(combined_data, ignore_index=True)
    combined_test_df = combined_test_df.sort_values('Date').reset_index(drop=True)
    
    combined_test_df['Error'] = combined_test_df['Prediction'] - combined_test_df['Actual']
    combined_test_df['Abs_Error'] = np.abs(combined_test_df['Error'])
    combined_test_df['Squared_Error'] = combined_test_df['Error'] ** 2
    
    overall_ic, overall_pval = calculate_ic(
        combined_test_df['Prediction'].values, 
        combined_test_df['Actual'].values, 
        method='spearman'
    )
    
    summary_stats = pd.DataFrame({
        'Metric': ['Sample Count', 'Overall IC', 'IC P-value', 'Mean Prediction', 'Mean Actual', 
                   'Std Prediction', 'Std Actual', 'MAE', 'MSE', 'RMSE'],
        'Value': [
            len(combined_test_df),
            overall_ic,
            overall_pval,
            combined_test_df['Prediction'].mean(),
            combined_test_df['Actual'].mean(),
            combined_test_df['Prediction'].std(),
            combined_test_df['Actual'].std(),
            combined_test_df['Abs_Error'].mean(),
            combined_test_df['Squared_Error'].mean(),
            np.sqrt(combined_test_df['Squared_Error'].mean())
        ]
    })
    
    excel_path = os.path.join(output_dir, 'combined_test_predictions.xlsx')
    
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        combined_test_df.to_excel(writer, sheet_name='测试集预测数据', index=False)
        summary_stats.to_excel(writer, sheet_name='统计摘要', index=False)
        
        fold_summary = combined_test_df.groupby('Fold').agg({
            'Prediction': ['mean', 'std', 'min', 'max'],
            'Actual': ['mean', 'std', 'min', 'max'],
            'Error': ['mean', 'std'],
            'Abs_Error': 'mean'
        }).round(6)
        fold_summary.to_excel(writer, sheet_name='各Fold统计')
    
    print(f"  已导出合并测试集数据到: {excel_path}")
    print(f"  总样本数: {len(combined_test_df)}")
    print(f"  时间范围: {combined_test_df['Date'].min().strftime('%Y-%m-%d')} 至 {combined_test_df['Date'].max().strftime('%Y-%m-%d')}")
    print(f"  整体测试集IC: {overall_ic:.4f} (p={overall_pval:.4f})")
    
    return combined_test_df, summary_stats

def create_summary_analysis(all_results, output_dir):
    """创建汇总分析"""
    print(f"\n创建汇总分析")
    print("=" * 50)
    
    summary_data = []
    for result in all_results:
        summary_data.append({
            'Fold': result['fold'],
            'Train_IC': result['train_ic'],
            'Test_IC': result['test_ic'],
            'Train_IC_PValue': result['train_ic_pval'],
            'Test_IC_PValue': result['test_ic_pval'],
            'IC_Significant': result['test_ic_pval'] < 0.05
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    # 计算统计指标
    print(f"各Fold IC结果:")
    for _, row in summary_df.iterrows():
        significance = "***" if row['Test_IC_PValue'] < 0.01 else "**" if row['Test_IC_PValue'] < 0.05 else "*" if row['Test_IC_PValue'] < 0.1 else ""
        print(f"  Fold {int(row['Fold']):02d}: 训练IC={row['Train_IC']:.4f}, 测试IC={row['Test_IC']:.4f}{significance} (p={row['Test_IC_PValue']:.4f})")
    
    print(f"\n整体IC统计:")
    print(f"平均IC (训练集): {summary_df['Train_IC'].mean():.4f} ± {summary_df['Train_IC'].std():.4f}")
    print(f"平均IC (测试集): {summary_df['Test_IC'].mean():.4f} ± {summary_df['Test_IC'].std():.4f}")
    print(f"IC显著性(p<0.05): {summary_df['IC_Significant'].sum()}/{len(summary_df)} ({summary_df['IC_Significant'].mean():.1%})")
    print(f"IC胜率: {(summary_df['Test_IC'] > 0).sum()}/{len(summary_df)} ({(summary_df['Test_IC'] > 0).mean():.1%})")
    
    # 创建全数据集IC分析
    overall_results = create_overall_ic_analysis(all_results, output_dir)
    
    # 导出合并测试集预测结果
    combined_test_df, summary_stats = export_combined_test_predictions(all_results, output_dir)
    
    # 保存结果
    with pd.ExcelWriter(os.path.join(output_dir, 'ic_analysis_results.xlsx')) as writer:
        summary_df.to_excel(writer, sheet_name='各Fold_IC结果', index=False)
    
    return summary_df

def create_overall_ic_analysis(all_results, output_dir):
    """创建全数据集的IC分析"""
    print(f"\n创建全数据集IC分析")
    print("=" * 50)
    
    # 拼接所有测试集的预测和实际值
    all_test_predictions = []
    all_test_actual = []
    all_train_predictions = []
    all_train_actual = []
    all_dates_test = []
    all_dates_train = []
    
    for result in all_results:
        all_test_predictions.extend(result['test_predictions'].tolist())
        all_test_actual.extend(result['test_actual'].tolist())
        all_train_predictions.extend(result['train_predictions'].tolist())
        all_train_actual.extend(result['train_actual'].tolist())
        all_dates_test.extend(result['test_predictions'].index.tolist())
        all_dates_train.extend(result['train_predictions'].index.tolist())
    
    # 计算全数据集IC
    overall_test_ic, overall_test_ic_pval = calculate_ic(all_test_predictions, all_test_actual, method='spearman')
    overall_train_ic, overall_train_ic_pval = calculate_ic(all_train_predictions, all_train_actual, method='spearman')
    
    # 输出结果
    print(f"全数据集IC分析:")
    print(f"训练集合并IC: {overall_train_ic:.4f} (p={overall_train_ic_pval:.4f})")
    print(f"测试集合并IC: {overall_test_ic:.4f} (p={overall_test_ic_pval:.4f})")
    print(f"测试集数据点数: {len(all_test_predictions)}")
    print(f"训练集数据点数: {len(all_train_predictions)}")
    
    if all_dates_test:
        print(f"测试集时间跨度: {min(all_dates_test).strftime('%Y-%m-%d')} 至 {max(all_dates_test).strftime('%Y-%m-%d')}")
    if all_dates_train:
        print(f"训练集时间跨度: {min(all_dates_train).strftime('%Y-%m-%d')} 至 {max(all_dates_train).strftime('%Y-%m-%d')}")
    
    # 保存全数据集结果
    overall_summary = pd.DataFrame({
        'Dataset': ['训练集合并', '测试集合并'],
        'IC': [overall_train_ic, overall_test_ic],
        'P_Value': [overall_train_ic_pval, overall_test_ic_pval],
        'Significant': [overall_train_ic_pval < 0.05, overall_test_ic_pval < 0.05],
        'Data_Points': [len(all_train_predictions), len(all_test_predictions)]
    })
    
    # 创建散点图
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            f'训练集预测vs实际 (IC: {overall_train_ic:.4f})',
            f'测试集预测vs实际 (IC: {overall_test_ic:.4f})'
        ]
    )
    
    # 训练集散点图
    fig.add_trace(go.Scatter(
        x=all_train_actual,
        y=all_train_predictions,
        mode='markers',
        name='训练集',
        marker=dict(color='blue', opacity=0.6, size=3),
        showlegend=False
    ), row=1, col=1)
    
    # 测试集散点图
    fig.add_trace(go.Scatter(
        x=all_test_actual,
        y=all_test_predictions,
        mode='markers',
        name='测试集',
        marker=dict(color='red', opacity=0.6, size=3),
        showlegend=False
    ), row=1, col=2)
    
    fig.update_layout(
        title='全数据集预测效果分析',
        height=500
    )
    
    fig.update_xaxes(title_text="实际值")
    fig.update_yaxes(title_text="预测值")
    
    # 保存结果
    with pd.ExcelWriter(os.path.join(output_dir, 'overall_ic_analysis.xlsx')) as writer:
        overall_summary.to_excel(writer, sheet_name='全数据集IC分析', index=False)
        
        # 保存详细数据
        overall_data = pd.DataFrame({
            'Date': all_dates_train + all_dates_test,
            'Type': ['Train'] * len(all_dates_train) + ['Test'] * len(all_dates_test),
            'Prediction': all_train_predictions + all_test_predictions,
            'Actual': all_train_actual + all_test_actual
        })
        overall_data.to_excel(writer, sheet_name='全数据集详细数据', index=False)
    
    fig.write_html(os.path.join(output_dir, 'overall_ic_scatter.html'))
    
    return {
        'overall_train_ic': overall_train_ic,
        'overall_test_ic': overall_test_ic,
        'summary': overall_summary
    }

def main():
    """主函数"""
    if not os.path.exists(DATA_PATH):
        print(f"错误：数据文件不存在 {DATA_PATH}")
        return
    
    # 加载稳定特征和超参数
    try:
        stable_features = load_stable_features(STABLE_FEATURES_FILE)
        fold_hyperparams = load_hyperparams(HYPERPARAMS_FILE)
    except FileNotFoundError as e:
        print(f"错误: {e}")
        print("请先运行原始分析代码生成稳定特征和超参数文件")
        return
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    available_folds = get_available_folds(DATA_PATH)
    
    print(f"XGBoost 稳定特征快速训练")
    print(f"发现 {len(available_folds)} 个fold")
    print(f"目标变量: {TARGET_NAME}")
    print(f"使用稳定特征数: {len(stable_features)}")
    print(f"IC计算方法: Spearman相关系数")
    print(f"策略: 使用稳定特征 + 保存的超参数，跳过特征选择和超参调优")
    
    all_results = []
    for fold_number in available_folds:
        try:
            result = process_single_fold(DATA_PATH, fold_number, TARGET_NAME, stable_features, fold_hyperparams, OUTPUT_DIR)
            if result:
                all_results.append(result)
        except Exception as e:
            print(f"Fold {fold_number} 处理失败: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    if not all_results:
        print("错误：没有成功处理任何fold")
        return
    
    # 创建汇总分析
    summary_df = create_summary_analysis(all_results, OUTPUT_DIR)
    
    print(f"\n分析完成！")
    print(f"成功处理 {len(all_results)} 个fold")
    print(f"结果保存至: {OUTPUT_DIR}")
    
    # 最终总结
    print(f"\n" + "="*60)
    print(f"最终IC总结:")
    print(f"="*60)
    print(f"各Fold平均测试IC: {summary_df['Test_IC'].mean():.4f} ± {summary_df['Test_IC'].std():.4f}")
    print(f"IC胜率: {(summary_df['Test_IC'] > 0).mean():.1%}")
    print(f"显著IC(p<0.05)占比: {summary_df['IC_Significant'].mean():.1%}")

if __name__ == "__main__":
    main()

从 QQQ_XGBoost_IC_Analysis/stable_features.txt 加载了 21 个稳定特征
从 QQQ_XGBoost_IC_Analysis/all_fold_hyperparams.json 加载了 12 个fold的超参数
XGBoost 稳定特征快速训练
发现 12 个fold
目标变量: 目标_未来10d回报率
使用稳定特征数: 21
IC计算方法: Spearman相关系数
策略: 使用稳定特征 + 保存的超参数，跳过特征选择和超参调优

处理 Fold 1
  使用保存的超参数:
    subsample: 0.9
    reg_lambda: 1
    reg_alpha: 0
    n_estimators: 400
    min_child_weight: 1
    max_depth: 5
    learning_rate: 0.03
    gamma: 0
    colsample_bytree: 0.7
  使用 21 个可用稳定特征
数据规模: 训练集 (1247, 21), 测试集 (252, 21)
训练集时间范围: 2009-01-02 至 2013-12-16
测试集时间范围: 2014-01-02 至 2014-12-31
  训练最终模型...
    训练集Spearman IC: 0.9802 (p=0.0000)
    测试集Spearman IC: 0.1832 (p=0.0035)
  已生成Fold 01的预测值分布图
  已生成Fold 01的散点图: QQQ_XGBoost_Stable_Features_Analysis(Regression)\Fold01\fold_01_scatter.html

处理 Fold 2
  使用保存的超参数:
    subsample: 0.9
    reg_lambda: 1
    reg_alpha: 0
    n_estimators: 400
    min_child_weight: 1
    max_depth: 5
    learning_rate: 0.03
    gamma: 0
    colsample_bytree: 0.7
  使用 21 个可用稳定特征
数据规模: 训练集 (1499, 

* **训练集**：平均IC (`0.7571`) ≈ 拼接后IC (`0.7540`) -> **表现一致**
* **测试集**：平均IC (`0.2736`) >> 拼接后IC (`0.0503`) -> **表现差异巨大**

这个现象并非偶然，也不是代码错误，而是由滚动验证（Rolling Validation）方法在训练集和测试集的数据构造方式上的根本不同所决定的。

---

### 第一部分：训练集IC表现一致的原理——“扩张与重叠”的稳定世界

让我们首先彻底搞清楚为什么训练集的两个IC值如此接近。这并非因为训练集数据是“独立同分布”的，而是因为您的代码设计，人为地创造了一个**高度稳定、同质化**的环境。

#### 1. 数据结构：扩张窗口 (Expanding Window)

您的代码通过`load_fold_data`和`get_available_folds`函数，为每个Fold生成训练集。这个过程是“扩张”的：
* **Fold 1** 的训练数据是 `2009-2013`年。
* **Fold 2** 的训练数据是 `2009-2014`年。
* **Fold 3** 的训练数据是 `2009-2015`年。
* ...
* **Fold 12** 的训练数据是 `2009-2024`年。

这种结构有一个极其重要的数学特性：**极高的数据重叠度**。
例如，Fold 2的训练集 (`~1500`个样本) 中，有大约`~1250`个样本是与Fold 1完全相同的，重叠度高达 **83%**。对于更靠后的Fold，这个重叠度会超过 **95%**。

#### 2. 对模型行为的影响：高度相似的模型

XGBoost模型是在这些数据上训练的。当您连续给两个模型输入几乎完全相同的“教材”时，它们必然会成长为两个**高度相似的模型**。
* 为Fold 10训练的模型，和为Fold 11训练的模型，它们内部的树结构、分裂点、特征权重都非常接近，因为它们的训练数据只有一年之差，但有十几年的数据是共享的。

#### 3. 对预测结果的影响：一致的预测“刻度尺”

当这些高度相似的模型，对它们各自的、高度重叠的训练集进行预测时（这是一个in-sample预测），它们输出的预测分，其**分布和含义（即我们讨论的“刻度尺”）**自然也是非常稳定和一致的。
* Fold 10的模型在`2009-2021`年数据上做出的预测，和Fold 11的模型在`2009-2022`年数据上做出的预测，它们在前13年的预测结果几乎是一样的。

#### 4. 最终的数学结果：无冲突的拼接

**这是关键**：当您将所有Fold的训练集预测结果拼接在一起时，由于数据源、模型、预测行为都高度一致，几乎**不会发生**我们在测试集中看到的、由于市场环境剧变导致的**“排名倒挂”**。
* 一个在Fold 10训练集中被预测为高分的样本，当它出现在Fold 11的训练集中时，它依然会被预测为高分。
* 因此，拼接后的“全局排名”与各个Fold的“局部排名”之间几乎没有冲突。

**结论**：由于训练集在滚动过程中的**“扩张与重叠”**特性，人为地创造了一个数据和模型都高度同质化的环境。这使得拼接操作不会引入系统性的排名冲突，所以其“拼接后IC”与“平均IC”必然非常接近。

---

### 第二部分：测试集IC差异巨大的原理——“独立与未知”的真实考验

现在，我们来看为什么测试集的表现完全不同。这是因为您的测试集构造方式，完美地模拟了真实世界。

#### 1. 数据结构：滚动窗口 (Rolling Window)，完全独立

您的测试集是逐年向前滚动的，并且**完全不重叠**：
* **Fold 1** 的测试集是 `2014`年。
* **Fold 2** 的测试集是 `2015`年。
* ...
* **Fold 12** 的测试集是 `2025`上半年。

这种结构意味着，每一个Fold的测试集都是一个**独立的、全新的、之前从未见过的市场环境**。2014年的平稳牛市，和2020年疫情冲击下的V型反转，是两个性质完全不同的“总体”。

#### 2. 对模型行为的影响：面对全新的、异质化的挑战

* **模型接受真实考验**：模型在训练阶段（比如`2009-2013`）从未见过`2014`年的市场规律。因此，测试集是对模型**泛化能力**的一次真实、无情的考验。
* **预测“刻度尺”的变化**：模型在面对不同市场环境时，其输出的预测分“刻度尺”会发生适应性变化。
    * 在`2017`年（低波动牛市），模型可能会给出一个范围在`[0.01, 0.08]`的预测分。
    * 在`2022`年（高波动熊市），模型可能会给出一个范围在`[-0.05, 0.02]`的预测分。

#### 3. 最终的数学结果：辛普森悖论与“排名倒挂”的出现

**这是导致巨大差异的直接原因**。
* **“平均测试IC” (`0.2736`)** 衡量的是模型在**每一次独立考验**中的平均表现。它告诉我们，您的模型适应能力很强，在每个独立的年份内，都能较好地完成排序任务。
* **“拼接后测试IC” (`0.0503`)** 是将这12次**风格迥异**的独立考验结果拼接后，进行**一次统一的全局排名**。这时，冲突就出现了：
    * 一个在`2022`年（熊市）被模型预测为最高分（例如`0.02`）的样本，它的真实收益率可能是`-1%`。
    * 一个在`2017`年（牛市）被模型预测为最低分（例如`0.01`）的样本，它的真实收益率可能是`+2%`。
    * 在**全局排名**中，模型会认为`0.02 > 0.01`，所以它会把前者的预测排在后面。但真实情况是`+2% > -1%`，后者的真实表现远好于前者。
    * 这就是一次**“排名倒挂”**。当成百上千次这样的、由市场环境切换导致的系统性“倒挂”累积起来，就会严重破坏全局预测排名和真实排名的一致性。

在数学上，每一次“排名倒挂”都会让斯皮尔曼相关系数公式中的“排名差异平方和”($\sum d_i^2$)增大，从而必然导致最终的IC值大幅降低。

---
 
### 总结

您观察到的现象，不仅正常，而且恰恰证明了您的回测框架是非常严谨和有效的。它清晰地区分了模型在“已知和相似”数据上的稳定拟合表现（训练集），和它在“未知和多变”的真实世界中的短期适应能力（平均测试IC）与长期稳健底线（拼接后测试IC）。

您好，这个问题提得太棒了，可以说一针见血，完全点出了这个操作最违反直觉、也最容易让人困惑的地方！

您的理解是完全正确的：对于同一个历史日期，比如`2013-10-10`，它确实对应着好几个不同的预测值，因为为Fold 1训练的模型和为Fold 2训练的模型是两个不同的模型。

那么代码到底是怎么把它们“拼”起来的呢？

答案是：它采用了一种非常简单粗暴、但用于诊断目的却非常有效的方式——**它并没有按日期去对齐或合并，而是直接将每一次的训练预测结果，作为一个独立的记录，全部追加到一个巨大的列表里。**

---

### 测试集“拼接”过程

让我们用一个最简单的例子来模拟您代码中`create_overall_ic_analysis`函数的行为：

**假设我们只有2个Fold：**

* **Fold 1 的运行过程:**
    1.  **训练集**: 包含日期 `D1, D2`。
    2.  **训练模型**: 得到模型 `M1`。
    3.  **进行预测**: 用`M1`对自己见过的训练集 `D1, D2` 进行预测，得到预测值 `P1, P2`。对应的真实值是 `A1, A2`。
    4.  **此时，代码生成了两个列表**:
        * `all_train_predictions = [P1, P2]`
        * `all_train_actuals = [A1, A2]`

* **Fold 2 的运行过程:**
    1.  **训练集**: 包含日期 `D1, D2, D3` (扩张了)。
    2.  **训练模型**: 得到一个**新的、更强的**模型 `M2`。
    3.  **进行预测**: 用`M2`对自己见过的训练集 `D1, D2, D3` 进行预测，得到预测值 `P1', P2', P3'`。对应的真实值是 `A1, A2, A3`。
    4.  **关键一步：追加 (Extend)**: 代码将这次的结果，**直接追加**到之前列表的末尾。
        * `all_train_predictions` 变成了 `[P1, P2,  **P1', P2', P3'**]`
        * `all_train_actuals`   变成了 `[A1, A2,  **A1, A2, A3**]`

**最终用于计算“拼接后训练IC”的数据是什么？**

最终，`scipy.stats.spearmanr`函数接收的是这两个长度为5的巨大列表。它在计算时，**完全不知道**第一个`A1`和第三个`A1`其实是同一个日期的真实收益。它只会机械地把这些数据看作是**5个独立的、不相关的样本对**：
1.  (P1, A1)
2.  (P2, A2)
3.  (P1', A1)
4.  (P2', A2)
5.  (P3', A3)

#### 为什么这样做，以及它的意义？


这个操作的**目的不是为了评估模型的预测能力**（评估能力要看完全独立的测试集），而是“一致性检验” (Consistency Check)。

* **它在检验什么？**
    它在检验整个**“滚动训练流程”的稳定性**。
* **我们看到了什么结果？**
    拼接后的IC (`0.7540`) 和平均IC (`0.7571`) 几乎一样。
* **这个结果说明了什么？**
    它有力地证明了：虽然模型`M1`, `M2`, `M3`...因为学习的数据不断增多而略有不同，但它们对于**同一段历史（例如2013年的数据）的“看法”和“判断模式”是高度一致的**。
    * 无论是由只学到2013年的`M1`来预测2013年的数据，还是由学到2020年的`M8`来回顾和预测2013年的数据，它们都能得出相似的、高IC的预测结果。

这极大地增强了我们对整个滚动回测框架**可靠性**的信心。它证明了您模型的学习过程是稳定的，不会因为新加入少量数据就发生剧烈的、不可预测的改变。


## 用保存的特征+重新调参+训练模型

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
import json
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

# ================================
# 配置参数
# ================================
DATA_PATH = 'QQQ_Rolling_Validation/QQQ_rolling_validation_dataset.xlsx'
OUTPUT_DIR = 'QQQ_XGBoost_Stable_Features_Tuning'
TARGET_NAME = '目标_未来10d回报率'

# 稳定特征文件路径
STABLE_FEATURES_FILE = 'QQQ_XGBoost_IC_Analysis/stable_features.txt'

# XGBoost参数网格
XGBOOST_PARAM_GRID = {
    'n_estimators': [100, 200, 300, 400],      
    'learning_rate': [0.01, 0.03, 0.05, 0.1], 
    'max_depth': [3, 4, 5, 6],                    
    'subsample': [0.7, 0.8, 0.9],                  
    'colsample_bytree': [0.7, 0.8, 0.9],           
    'reg_alpha': [0, 0.1, 0.5],                    
    'reg_lambda': [1, 1.5, 2],                     
    'min_child_weight': [1, 3, 5],                 
    'gamma': [0, 0.1, 0.2]                         
}

# 策略参数
RANDOM_SEARCH_ITER = 100
SHAP_SAMPLE_SIZE = 500

# ================================
# 核心函数
# ================================

def set_chinese_font():
    """设置中文字体"""
    try:
        plt.rcParams['font.sans-serif'] = ['SimHei']
        plt.rcParams['axes.unicode_minus'] = False
    except:
        pass

def load_stable_features(file_path=STABLE_FEATURES_FILE):
    """加载稳定特征列表"""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"稳定特征文件不存在: {file_path}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        features = [line.strip() for line in f if line.strip()]
    
    print(f"从 {file_path} 加载了 {len(features)} 个稳定特征")
    return features

def get_available_folds(data_path):
    """获取所有可用的fold编号"""
    excel_file = pd.ExcelFile(data_path)
    fold_sheets = [sheet for sheet in excel_file.sheet_names if sheet.startswith('Fold') and sheet.endswith('_训练集')]
    fold_numbers = [int(sheet.split('Fold')[1].split('_')[0]) for sheet in fold_sheets]
    return sorted(fold_numbers)

def load_fold_data(data_path, fold_number, target_name, stable_features):
    """加载fold数据，只使用稳定特征"""
    train_sheet = f'Fold{fold_number:02d}_训练集'
    train_df = pd.read_excel(data_path, sheet_name=train_sheet, index_col=0, parse_dates=True)
    train_df = train_df.dropna()
    
    test_sheet = f'Fold{fold_number:02d}_测试集'
    test_df = pd.read_excel(data_path, sheet_name=test_sheet, index_col=0, parse_dates=True)
    test_df = test_df.dropna()
    
    # 检查稳定特征是否存在于数据中
    available_features = [f for f in stable_features if f in train_df.columns]
    missing_features = [f for f in stable_features if f not in train_df.columns]
    
    if missing_features:
        print(f"  警告: {len(missing_features)} 个稳定特征在数据中不存在")
        if len(missing_features) <= 5:
            print(f"  缺失特征: {missing_features}")
    
    print(f"  使用 {len(available_features)} 个可用稳定特征")
    
    # 使用可用的稳定特征
    X_train = train_df[available_features]
    y_train = train_df[target_name]
    X_test = test_df[available_features]
    y_test = test_df[target_name]
    
    return X_train, y_train, X_test, y_test, available_features

def calculate_ic(predictions, targets, method='spearman'):
    """计算IC值"""
    if method == 'pearson':
        ic, p_value = stats.pearsonr(predictions, targets)
    elif method == 'spearman':
        ic, p_value = stats.spearmanr(predictions, targets)
    return ic, p_value

class DynamicTimeSeriesSplit:
    """动态时间序列分割，根据fold自动调整分割数量"""
    
    def __init__(self, fold_number=1):
        self.fold_number = fold_number
    
    def split(self, X, y=None, groups=None):
        """
        根据fold动态调整分割策略
        """
        # 获取所有年份
        years = sorted(X.index.year.unique())
        
        print(f"    动态TimeSeriesSplit调参过程 (Fold {self.fold_number}):")
        print(f"    可用年份: {years}")
        print(f"    总数据量: {len(X)} 个样本")
        
        splits = []
        
        # 确保有足够的年份进行交叉验证
        if len(years) < 3:
            print(f"    警告: 年份数量不足({len(years)}年)，无法进行时间序列交叉验证")
            return splits
        
        # 动态计算splits数量：fold1有3个split，fold2有4个split，以此类推
        # 但不能超过可用年份数量的限制
        target_splits = 2 + self.fold_number  # fold1->3, fold2->4, fold3->5...
        max_possible_splits = len(years) - 1   # 至少需要1年做验证集
        actual_splits = min(target_splits, max_possible_splits)
        
        print(f"    目标splits数: {target_splits}, 实际splits数: {actual_splits}")
        
        for i in range(actual_splits):
            # 训练集：从第一年到第(i+1)年
            train_years = years[:i+2]  # i=0时取前2年，i=1时取前3年...
            
            # 验证集：第(i+2)年的下一年
            val_year_index = i + 2
            if val_year_index < len(years):
                val_year = years[val_year_index]
            else:
                break
            
            # 获取训练集数据
            train_mask = X.index.year.isin(train_years)
            train_indices = X.index[train_mask]
            
            # 从训练集中去掉最后10个交易日
            if len(train_indices) > 10:
                train_indices = train_indices[:-10]
            
            # 获取验证集数据
            val_mask = X.index.year == val_year
            val_indices = X.index[val_mask]
            
            if len(train_indices) > 0 and len(val_indices) > 0:
                # 转换为位置索引
                train_pos = [X.index.get_loc(idx) for idx in train_indices]
                val_pos = [X.index.get_loc(idx) for idx in val_indices]
                
                print(f"    Split {i+1}:")
                print(f"      训练集: {train_years} (去掉最后10个交易日)")
                print(f"      训练集时间: {train_indices[0].strftime('%Y-%m-%d')} 至 {train_indices[-1].strftime('%Y-%m-%d')} ({len(train_indices)} 样本)")
                print(f"      验证集: {val_year}年")
                print(f"      验证集时间: {val_indices[0].strftime('%Y-%m-%d')} 至 {val_indices[-1].strftime('%Y-%m-%d')} ({len(val_indices)} 样本)")
                
                splits.append((train_pos, val_pos))
        
        print(f"    总共生成 {len(splits)} 个splits")
        return splits
    
    def get_n_splits(self, X=None, y=None, groups=None):
        if X is not None:
            years = sorted(X.index.year.unique())
            target_splits = 2 + self.fold_number
            max_possible_splits = len(years) - 1
            return min(target_splits, max_possible_splits)
        return 2 + self.fold_number

def ic_scorer(y_true, y_pred):
    """IC评分函数，用于交叉验证"""
    ic, _ = calculate_ic(y_pred, y_true, method='spearman')
    return ic

def hyperparameter_tuning(X_train, y_train, fold_number):
    """超参数调优"""
    print("  进行超参数调优...")
    
    base_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1)
    tscv = DynamicTimeSeriesSplit(fold_number=fold_number)
    
    # 创建IC评分器
    ic_scorer_func = make_scorer(ic_scorer, greater_is_better=True)
    
    random_search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=XGBOOST_PARAM_GRID,
        n_iter=RANDOM_SEARCH_ITER,
        cv=tscv,
        scoring=ic_scorer_func,  # 使用IC作为评分标准
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    
    random_search.fit(X_train, y_train)
    
    print(f"    最佳CV IC得分: {random_search.best_score_:.6f}")
    print(f"    使用的CV splits数: {tscv.get_n_splits(X_train)}")
    return random_search.best_params_

def train_final_model(X_train, y_train, X_test, y_test, best_params):
    """训练最终模型并计算IC"""
    print("  训练最终模型...")
    
    final_model = xgb.XGBRegressor(**best_params, random_state=42, n_jobs=-1)
    final_model.fit(X_train, y_train)
    
    y_train_pred = final_model.predict(X_train)
    y_test_pred = final_model.predict(X_test)
    
    # 转换为Series并保持原索引
    y_train_pred = pd.Series(y_train_pred, index=y_train.index)
    y_test_pred = pd.Series(y_test_pred, index=y_test.index)
    
    # 计算IC (使用Spearman相关系数)
    train_ic, train_ic_pval = calculate_ic(y_train_pred.values, y_train.values, method='spearman')
    test_ic, test_ic_pval = calculate_ic(y_test_pred.values, y_test.values, method='spearman')
    
    print(f"    训练集Spearman IC: {train_ic:.4f} (p={train_ic_pval:.4f})")
    print(f"    测试集Spearman IC: {test_ic:.4f} (p={test_ic_pval:.4f})")
    
    return final_model, y_train_pred, y_test_pred, train_ic, test_ic, train_ic_pval, test_ic_pval

def create_prediction_distribution_plot(y_train_pred, y_test_pred, y_train, y_test, fold_number, fold_dir):
    """为每个fold创建预测值分布图"""
    set_chinese_font()
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 训练集预测值分布
    axes[0, 0].hist(y_train_pred, bins=30, alpha=0.7, color='blue', edgecolor='black', label='预测值')
    axes[0, 0].axvline(y_train_pred.mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {y_train_pred.mean():.4f}')
    axes[0, 0].set_title(f'Fold {fold_number:02d} - 训练集预测值分布')
    axes[0, 0].set_xlabel('预测值')
    axes[0, 0].set_ylabel('频数')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 测试集预测值分布
    axes[0, 1].hist(y_test_pred, bins=30, alpha=0.7, color='green', edgecolor='black', label='预测值')
    axes[0, 1].axvline(y_test_pred.mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {y_test_pred.mean():.4f}')
    axes[0, 1].set_title(f'Fold {fold_number:02d} - 测试集预测值分布')
    axes[0, 1].set_xlabel('预测值')
    axes[0, 1].set_ylabel('频数')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 训练集：预测值vs真实值对比分布
    axes[1, 0].hist(y_train, bins=30, alpha=0.5, color='blue', edgecolor='black', label='真实值')
    axes[1, 0].hist(y_train_pred, bins=30, alpha=0.5, color='red', edgecolor='black', label='预测值')
    axes[1, 0].set_title(f'Fold {fold_number:02d} - 训练集预测vs真实分布')
    axes[1, 0].set_xlabel('值')
    axes[1, 0].set_ylabel('频数')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 测试集：预测值vs真实值对比分布
    axes[1, 1].hist(y_test, bins=30, alpha=0.5, color='blue', edgecolor='black', label='真实值')
    axes[1, 1].hist(y_test_pred, bins=30, alpha=0.5, color='red', edgecolor='black', label='预测值')
    axes[1, 1].set_title(f'Fold {fold_number:02d} - 测试集预测vs真实分布')
    axes[1, 1].set_xlabel('值')
    axes[1, 1].set_ylabel('频数')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    fig.suptitle(f'Fold {fold_number:02d} 预测值分布分析', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(fold_dir, f'prediction_distribution.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  已生成Fold {fold_number:02d}的预测值分布图")

def create_fold_scatter_plot(y_train_pred, y_test_pred, y_train, y_test, fold_number, fold_dir):
    """为每个fold创建散点图"""
    # 计算IC
    train_ic, train_ic_pval = calculate_ic(y_train_pred.values, y_train.values, method='spearman')
    test_ic, test_ic_pval = calculate_ic(y_test_pred.values, y_test.values, method='spearman')
    
    # 创建散点图
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            f'训练集预测vs实际 (IC: {train_ic:.4f})',
            f'测试集预测vs实际 (IC: {test_ic:.4f})'
        ]
    )
    
    # 训练集散点图
    fig.add_trace(go.Scatter(
        x=y_train.values,
        y=y_train_pred.values,
        mode='markers',
        name='训练集',
        marker=dict(color='blue', opacity=0.6, size=3),
        showlegend=False
    ), row=1, col=1)
    
    # 测试集散点图
    fig.add_trace(go.Scatter(
        x=y_test.values,
        y=y_test_pred.values,
        mode='markers',
        name='测试集',
        marker=dict(color='red', opacity=0.6, size=3),
        showlegend=False
    ), row=1, col=2)
    
    fig.update_layout(
        title=f'Fold {fold_number:02d} 预测效果分析',
        height=500
    )
    
    fig.update_xaxes(title_text="实际值")
    fig.update_yaxes(title_text="预测值")
    
    # 保存图片
    scatter_file = os.path.join(fold_dir, f'fold_{fold_number:02d}_scatter.html')
    fig.write_html(scatter_file)
    
    print(f"  已生成Fold {fold_number:02d}的散点图: {scatter_file}")

def process_single_fold(data_path, fold_number, target_name, stable_features, output_dir):
    """处理单个fold"""
    print(f"\n处理 Fold {fold_number}")
    print("=" * 50)
    
    # 加载数据，使用稳定特征
    X_train, y_train, X_test, y_test, used_features = load_fold_data(data_path, fold_number, target_name, stable_features)
    
    print(f"数据规模: 训练集 {X_train.shape}, 测试集 {X_test.shape}")
    print(f"训练集时间范围: {X_train.index[0].strftime('%Y-%m-%d')} 至 {X_train.index[-1].strftime('%Y-%m-%d')}")
    print(f"测试集时间范围: {X_test.index[0].strftime('%Y-%m-%d')} 至 {X_test.index[-1].strftime('%Y-%m-%d')}")
    
    # 超参数调优（针对稳定特征进行调优）
    best_params = hyperparameter_tuning(X_train, y_train, fold_number)
    
    # 训练最终模型
    final_model, y_train_pred, y_test_pred, train_ic, test_ic, train_ic_pval, test_ic_pval = train_final_model(
        X_train, y_train, X_test, y_test, best_params)
    
    # 输出最优超参数
    print(f"  最优超参数:")
    for param, value in best_params.items():
        print(f"    {param}: {value}")
    
    # 保存结果
    fold_dir = os.path.join(output_dir, f'Fold{fold_number:02d}')
    os.makedirs(fold_dir, exist_ok=True)
    
    # 保存预测结果
    results_df = pd.DataFrame({
        'Date': y_train.index.tolist() + y_test.index.tolist(),
        'Type': ['Train'] * len(y_train) + ['Test'] * len(y_test),
        'Prediction': y_train_pred.tolist() + y_test_pred.tolist(),
        'Actual': y_train.tolist() + y_test.tolist()
    })
    results_df.to_csv(os.path.join(fold_dir, 'predictions.csv'), index=False)
    
    # 保存使用的特征列表
    pd.DataFrame({'Feature': used_features}).to_csv(os.path.join(fold_dir, 'used_features.csv'), index=False)
    
    # 保存优化后的超参数
    with open(os.path.join(fold_dir, 'best_hyperparams.json'), 'w', encoding='utf-8') as f:
        json.dump(best_params, f, indent=2, ensure_ascii=False)
    
    # 创建预测值分布图
    create_prediction_distribution_plot(y_train_pred, y_test_pred, y_train, y_test, fold_number, fold_dir)
    
    # 创建散点图
    create_fold_scatter_plot(y_train_pred, y_test_pred, y_train, y_test, fold_number, fold_dir)
    
    return {
        'fold': fold_number,
        'train_ic': train_ic,
        'test_ic': test_ic,
        'train_ic_pval': train_ic_pval,
        'test_ic_pval': test_ic_pval,
        'used_features': used_features,
        'best_params': best_params,
        'test_predictions': y_test_pred,
        'test_actual': y_test,
        'train_predictions': y_train_pred,
        'train_actual': y_train
    }

def export_combined_test_predictions(all_results, output_dir):
    """导出所有fold测试集拼起来的预测值和真实值到Excel"""
    print(f"\n导出合并测试集预测结果到Excel")
    print("=" * 50)
    
    combined_data = []
    
    for result in all_results:
        fold_number = result['fold']
        test_predictions = result['test_predictions']
        test_actual = result['test_actual']
        
        fold_df = pd.DataFrame({
            'Date': test_predictions.index,
            'Fold': fold_number,
            'Prediction': test_predictions.values,
            'Actual': test_actual.values
        })
        
        combined_data.append(fold_df)
    
    combined_test_df = pd.concat(combined_data, ignore_index=True)
    combined_test_df = combined_test_df.sort_values('Date').reset_index(drop=True)
    
    combined_test_df['Error'] = combined_test_df['Prediction'] - combined_test_df['Actual']
    combined_test_df['Abs_Error'] = np.abs(combined_test_df['Error'])
    combined_test_df['Squared_Error'] = combined_test_df['Error'] ** 2
    
    overall_ic, overall_pval = calculate_ic(
        combined_test_df['Prediction'].values, 
        combined_test_df['Actual'].values, 
        method='spearman'
    )
    
    summary_stats = pd.DataFrame({
        'Metric': ['Sample Count', 'Overall IC', 'IC P-value', 'Mean Prediction', 'Mean Actual', 
                   'Std Prediction', 'Std Actual', 'MAE', 'MSE', 'RMSE'],
        'Value': [
            len(combined_test_df),
            overall_ic,
            overall_pval,
            combined_test_df['Prediction'].mean(),
            combined_test_df['Actual'].mean(),
            combined_test_df['Prediction'].std(),
            combined_test_df['Actual'].std(),
            combined_test_df['Abs_Error'].mean(),
            combined_test_df['Squared_Error'].mean(),
            np.sqrt(combined_test_df['Squared_Error'].mean())
        ]
    })
    
    excel_path = os.path.join(output_dir, 'combined_test_predictions.xlsx')
    
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        combined_test_df.to_excel(writer, sheet_name='测试集预测数据', index=False)
        summary_stats.to_excel(writer, sheet_name='统计摘要', index=False)
        
        fold_summary = combined_test_df.groupby('Fold').agg({
            'Prediction': ['mean', 'std', 'min', 'max'],
            'Actual': ['mean', 'std', 'min', 'max'],
            'Error': ['mean', 'std'],
            'Abs_Error': 'mean'
        }).round(6)
        fold_summary.to_excel(writer, sheet_name='各Fold统计')
    
    print(f"  已导出合并测试集数据到: {excel_path}")
    print(f"  总样本数: {len(combined_test_df)}")
    print(f"  时间范围: {combined_test_df['Date'].min().strftime('%Y-%m-%d')} 至 {combined_test_df['Date'].max().strftime('%Y-%m-%d')}")
    print(f"  整体测试集IC: {overall_ic:.4f} (p={overall_pval:.4f})")
    
    return combined_test_df, summary_stats

def create_summary_analysis(all_results, output_dir):
    """创建汇总分析"""
    print(f"\n创建汇总分析")
    print("=" * 50)
    
    summary_data = []
    for result in all_results:
        summary_data.append({
            'Fold': result['fold'],
            'Train_IC': result['train_ic'],
            'Test_IC': result['test_ic'],
            'Train_IC_PValue': result['train_ic_pval'],
            'Test_IC_PValue': result['test_ic_pval'],
            'IC_Significant': result['test_ic_pval'] < 0.05
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    # 计算统计指标
    print(f"各Fold IC结果:")
    for _, row in summary_df.iterrows():
        significance = "***" if row['Test_IC_PValue'] < 0.01 else "**" if row['Test_IC_PValue'] < 0.05 else "*" if row['Test_IC_PValue'] < 0.1 else ""
        print(f"  Fold {int(row['Fold']):02d}: 训练IC={row['Train_IC']:.4f}, 测试IC={row['Test_IC']:.4f}{significance} (p={row['Test_IC_PValue']:.4f})")
    
    print(f"\n整体IC统计:")
    print(f"平均IC (训练集): {summary_df['Train_IC'].mean():.4f} ± {summary_df['Train_IC'].std():.4f}")
    print(f"平均IC (测试集): {summary_df['Test_IC'].mean():.4f} ± {summary_df['Test_IC'].std():.4f}")
    print(f"IC显著性(p<0.05): {summary_df['IC_Significant'].sum()}/{len(summary_df)} ({summary_df['IC_Significant'].mean():.1%})")
    print(f"IC胜率: {(summary_df['Test_IC'] > 0).sum()}/{len(summary_df)} ({(summary_df['Test_IC'] > 0).mean():.1%})")
    
    # 创建全数据集IC分析
    overall_results = create_overall_ic_analysis(all_results, output_dir)
    
    # 导出合并测试集预测结果
    combined_test_df, summary_stats = export_combined_test_predictions(all_results, output_dir)
    
    # 保存结果
    with pd.ExcelWriter(os.path.join(output_dir, 'ic_analysis_results.xlsx')) as writer:
        summary_df.to_excel(writer, sheet_name='各Fold_IC结果', index=False)
    
    # 保存新的超参数配置
    save_optimized_hyperparams(all_results, output_dir)
    
    return summary_df

def create_overall_ic_analysis(all_results, output_dir):
    """创建全数据集的IC分析"""
    print(f"\n创建全数据集IC分析")
    print("=" * 50)
    
    # 拼接所有测试集的预测和实际值
    all_test_predictions = []
    all_test_actual = []
    all_train_predictions = []
    all_train_actual = []
    all_dates_test = []
    all_dates_train = []
    
    for result in all_results:
        all_test_predictions.extend(result['test_predictions'].tolist())
        all_test_actual.extend(result['test_actual'].tolist())
        all_train_predictions.extend(result['train_predictions'].tolist())
        all_train_actual.extend(result['train_actual'].tolist())
        all_dates_test.extend(result['test_predictions'].index.tolist())
        all_dates_train.extend(result['train_predictions'].index.tolist())
    
    # 计算全数据集IC
    overall_test_ic, overall_test_ic_pval = calculate_ic(all_test_predictions, all_test_actual, method='spearman')
    overall_train_ic, overall_train_ic_pval = calculate_ic(all_train_predictions, all_train_actual, method='spearman')
    
    # 输出结果
    print(f"全数据集IC分析:")
    print(f"训练集合并IC: {overall_train_ic:.4f} (p={overall_train_ic_pval:.4f})")
    print(f"测试集合并IC: {overall_test_ic:.4f} (p={overall_test_ic_pval:.4f})")
    print(f"测试集数据点数: {len(all_test_predictions)}")
    print(f"训练集数据点数: {len(all_train_predictions)}")
    
    if all_dates_test:
        print(f"测试集时间跨度: {min(all_dates_test).strftime('%Y-%m-%d')} 至 {max(all_dates_test).strftime('%Y-%m-%d')}")
    if all_dates_train:
        print(f"训练集时间跨度: {min(all_dates_train).strftime('%Y-%m-%d')} 至 {max(all_dates_train).strftime('%Y-%m-%d')}")
    
    # 保存全数据集结果
    overall_summary = pd.DataFrame({
        'Dataset': ['训练集合并', '测试集合并'],
        'IC': [overall_train_ic, overall_test_ic],
        'P_Value': [overall_train_ic_pval, overall_test_ic_pval],
        'Significant': [overall_train_ic_pval < 0.05, overall_test_ic_pval < 0.05],
        'Data_Points': [len(all_train_predictions), len(all_test_predictions)]
    })
    
    # 创建散点图
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            f'训练集预测vs实际 (IC: {overall_train_ic:.4f})',
            f'测试集预测vs实际 (IC: {overall_test_ic:.4f})'
        ]
    )
    
    # 训练集散点图
    fig.add_trace(go.Scatter(
        x=all_train_actual,
        y=all_train_predictions,
        mode='markers',
        name='训练集',
        marker=dict(color='blue', opacity=0.6, size=3),
        showlegend=False
    ), row=1, col=1)
    
    # 测试集散点图
    fig.add_trace(go.Scatter(
        x=all_test_actual,
        y=all_test_predictions,
        mode='markers',
        name='测试集',
        marker=dict(color='red', opacity=0.6, size=3),
        showlegend=False
    ), row=1, col=2)
    
    fig.update_layout(
        title='全数据集预测效果分析',
        height=500
    )
    
    fig.update_xaxes(title_text="实际值")
    fig.update_yaxes(title_text="预测值")
    
    # 保存结果
    with pd.ExcelWriter(os.path.join(output_dir, 'overall_ic_analysis.xlsx')) as writer:
        overall_summary.to_excel(writer, sheet_name='全数据集IC分析', index=False)
        
        # 保存详细数据
        overall_data = pd.DataFrame({
            'Date': all_dates_train + all_dates_test,
            'Type': ['Train'] * len(all_dates_train) + ['Test'] * len(all_dates_test),
            'Prediction': all_train_predictions + all_test_predictions,
            'Actual': all_train_actual + all_test_actual
        })
        overall_data.to_excel(writer, sheet_name='全数据集详细数据', index=False)
    
    fig.write_html(os.path.join(output_dir, 'overall_ic_scatter.html'))
    
    return {
        'overall_train_ic': overall_train_ic,
        'overall_test_ic': overall_test_ic,
        'summary': overall_summary
    }

def save_optimized_hyperparams(all_results, output_dir):
    """保存针对稳定特征优化后的超参数"""
    print(f"\n保存优化后的超参数")
    print("=" * 50)
    
    # 保存每个fold的超参数
    hyperparams_data = []
    for result in all_results:
        fold_params = {'Fold': result['fold']}
        fold_params.update(result['best_params'])
        hyperparams_data.append(fold_params)
    
    # 保存为JSON
    hyperparams_json = os.path.join(output_dir, 'optimized_hyperparams_for_stable_features.json')
    with open(hyperparams_json, 'w', encoding='utf-8') as f:
        json.dump(hyperparams_data, f, indent=2, ensure_ascii=False)
    
    print(f"已保存优化后的超参数到: {hyperparams_json}")
    
    # 分析超参数分布
    param_analysis = {}
    for param in XGBOOST_PARAM_GRID.keys():
        values = [result['best_params'][param] for result in all_results if param in result['best_params']]
        if values:
            param_analysis[param] = {
                'mean': np.mean(values),
                'std': np.std(values),
                'min': np.min(values),
                'max': np.max(values),
                'most_common': max(set(values), key=values.count)
            }
    
    # 保存参数分析
    param_analysis_df = pd.DataFrame(param_analysis).T
    param_analysis_df.to_csv(os.path.join(output_dir, 'hyperparameter_analysis.csv'))
    
    print(f"已保存超参数分析到: {os.path.join(output_dir, 'hyperparameter_analysis.csv')}")
    
    return hyperparams_data, param_analysis

def create_ic_visualization(summary_df, output_dir):
    """创建IC可视化"""
    set_chinese_font()
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # IC分布直方图
    axes[0, 0].hist(summary_df['Test_IC'], bins=10, alpha=0.7, color='blue', edgecolor='black')
    axes[0, 0].axvline(summary_df['Test_IC'].mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {summary_df["Test_IC"].mean():.3f}')
    axes[0, 0].axvline(0, color='black', linestyle='-', alpha=0.5)
    axes[0, 0].set_title('测试集IC分布')
    axes[0, 0].set_xlabel('IC值')
    axes[0, 0].set_ylabel('频数')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 各Fold IC对比
    x_pos = range(len(summary_df))
    colors = ['green' if ic > 0 else 'red' for ic in summary_df['Test_IC']]
    bars = axes[0, 1].bar(x_pos, summary_df['Test_IC'], color=colors, alpha=0.7)
    axes[0, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
    axes[0, 1].axhline(y=summary_df['Test_IC'].mean(), color='blue', linestyle='--', linewidth=2, label=f'平均IC: {summary_df["Test_IC"].mean():.3f}')
    axes[0, 1].set_title('各Fold测试集IC')
    axes[0, 1].set_xlabel('Fold')
    axes[0, 1].set_ylabel('IC值')
    axes[0, 1].set_xticks(x_pos)
    axes[0, 1].set_xticklabels([f'F{int(fold):02d}' for fold in summary_df['Fold']])
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 添加显著性标记
    for i, (bar, pval) in enumerate(zip(bars, summary_df['Test_IC_PValue'])):
        height = bar.get_height()
        if pval < 0.01:
            axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + (0.01 if height >= 0 else -0.02), 
                          '***', ha='center', va='bottom' if height >= 0 else 'top', fontsize=12, fontweight='bold')
        elif pval < 0.05:
            axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + (0.01 if height >= 0 else -0.02), 
                          '**', ha='center', va='bottom' if height >= 0 else 'top', fontsize=12, fontweight='bold')
        elif pval < 0.1:
            axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + (0.01 if height >= 0 else -0.02), 
                          '*', ha='center', va='bottom' if height >= 0 else 'top', fontsize=12, fontweight='bold')
    
    # 训练vs测试IC对比
    axes[1, 0].scatter(summary_df['Train_IC'], summary_df['Test_IC'], alpha=0.7, s=60, color='blue')
    axes[1, 0].plot([-0.3, 0.3], [-0.3, 0.3], 'r--', alpha=0.5, label='y=x')
    axes[1, 0].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    axes[1, 0].axvline(x=0, color='black', linestyle='-', alpha=0.3)
    axes[1, 0].set_xlabel('训练集IC')
    axes[1, 0].set_ylabel('测试集IC')
    axes[1, 0].set_title('训练集vs测试集IC')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # IC p值分布
    axes[1, 1].hist(summary_df['Test_IC_PValue'], bins=10, alpha=0.7, color='orange', edgecolor='black')
    axes[1, 1].axvline(0.05, color='red', linestyle='--', linewidth=2, label='p=0.05')
    axes[1, 1].axvline(0.01, color='darkred', linestyle='--', linewidth=2, label='p=0.01')
    axes[1, 1].set_title('测试集IC p值分布')
    axes[1, 1].set_xlabel('p值')
    axes[1, 1].set_ylabel('频数')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'ic_analysis_summary.png'), dpi=300, bbox_inches='tight')
    plt.close()

def main():
    """主函数"""
    if not os.path.exists(DATA_PATH):
        print(f"错误：数据文件不存在 {DATA_PATH}")
        return
    
    # 加载稳定特征
    try:
        stable_features = load_stable_features(STABLE_FEATURES_FILE)
    except FileNotFoundError as e:
        print(f"错误: {e}")
        print("请先运行原始分析代码生成稳定特征文件")
        return
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    available_folds = get_available_folds(DATA_PATH)
    
    print(f"XGBoost 稳定特征超参数调优训练")
    print(f"发现 {len(available_folds)} 个fold")
    print(f"目标变量: {TARGET_NAME}")
    print(f"使用稳定特征数: {len(stable_features)}")
    print(f"随机搜索迭代次数: {RANDOM_SEARCH_ITER}")
    print(f"IC计算方法: Spearman相关系数")
    print(f"策略: 使用稳定特征 + 重新进行超参数调优")
    
    all_results = []
    for fold_number in available_folds:
        try:
            result = process_single_fold(DATA_PATH, fold_number, TARGET_NAME, stable_features, OUTPUT_DIR)
            if result:
                all_results.append(result)
        except Exception as e:
            print(f"Fold {fold_number} 处理失败: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    if not all_results:
        print("错误：没有成功处理任何fold")
        return
    
    # 创建汇总分析
    summary_df = create_summary_analysis(all_results, OUTPUT_DIR)
    
    # 创建可视化
    create_ic_visualization(summary_df, OUTPUT_DIR)
    
    print(f"\n分析完成！")
    print(f"成功处理 {len(all_results)} 个fold")
    print(f"结果保存至: {OUTPUT_DIR}")
    
    # 最终总结
    print(f"\n" + "="*60)
    print(f"最终IC总结:")
    print(f"="*60)
    print(f"各Fold平均测试IC: {summary_df['Test_IC'].mean():.4f} ± {summary_df['Test_IC'].std():.4f}")
    print(f"IC胜率: {(summary_df['Test_IC'] > 0).mean():.1%}")
    print(f"显著IC(p<0.05)占比: {summary_df['IC_Significant'].mean():.1%}")
    print(f"\n已保存:")
    print(f"- 针对稳定特征优化的超参数配置")
    print(f"- 完整分析结果")
    print(f"- 各种可视化图表")

if __name__ == "__main__":
    main()

从 QQQ_XGBoost_IC_Analysis/stable_features.txt 加载了 21 个稳定特征
XGBoost 稳定特征超参数调优训练
发现 12 个fold
目标变量: 目标_未来10d回报率
使用稳定特征数: 21
随机搜索迭代次数: 100
IC计算方法: Spearman相关系数
策略: 使用稳定特征 + 重新进行超参数调优

处理 Fold 1
  使用 21 个可用稳定特征
数据规模: 训练集 (1247, 21), 测试集 (252, 21)
训练集时间范围: 2009-01-02 至 2013-12-16
测试集时间范围: 2014-01-02 至 2014-12-31
  进行超参数调优...
    动态TimeSeriesSplit调参过程 (Fold 1):
    可用年份: [2009, 2010, 2011, 2012, 2013]
    总数据量: 1247 个样本
    目标splits数: 3, 实际splits数: 3
    Split 1:
      训练集: [2009, 2010] (去掉最后10个交易日)
      训练集时间: 2009-01-02 至 2010-12-16 (494 样本)
      验证集: 2011年
      验证集时间: 2011-01-03 至 2011-12-30 (252 样本)
    Split 2:
      训练集: [2009, 2010, 2011] (去掉最后10个交易日)
      训练集时间: 2009-01-02 至 2011-12-15 (746 样本)
      验证集: 2012年
      验证集时间: 2012-01-03 至 2012-12-31 (249 样本)
    Split 3:
      训练集: [2009, 2010, 2011, 2012] (去掉最后10个交易日)
      训练集时间: 2009-01-02 至 2012-12-14 (995 样本)
      验证集: 2013年
      验证集时间: 2013-01-02 至 2013-12-16 (242 样本)
    总共生成 3 个splits
    最佳CV IC得分: 0.110563
    使用的CV spli

## 固定一组XGB参数+归一化对比 ##
只有当‘预测值’和‘真实值’的分布，都随着市场环境（Fold）发生同方向的、清晰的分层，且不同Fold之间的范围没有显著重叠或倒挂时，‘拼接后IC’才会约等于‘平均IC’。

---

### 问题陈述：一个看似矛盾的核心观察

首先，我们必须清晰地定义我们所观察到的现象。在您的滚动回测结果中，存在两种截然不同的模式：

1.  **在训练集上 (In-Sample Performance)**：
    * **“各Fold IC的平均值”** (`0.7571`) 与 **“所有Fold拼接后的总IC”** (`0.7540`) **基本相等**。这表明，在模型已经“见过”的数据上，其表现是高度一致和稳定的。

2.  **在测试集上 (Out-of-Sample Performance)**：
    * **“各Fold IC的平均值”** (`0.2736`) 远高于 **“所有Fold拼接后的总IC”** (`0.0503`)，两者**差异巨大**。这表明，在模型从未见过的未知数据上，其表现模式发生了根本性的变化。

这个现象并非偶然，也不是代码错误，而是由您严谨的**滚动验证（Rolling Validation）**方法所揭示的一个深刻的、关于金融时间序列预测的本质规律。

---

### 第一部分：根本性原理 —— 辛普森悖论与非平稳性

要理解这个现象，我们必须首先引入一个核心的统计学概念：**辛普森悖论 (Simpson's Paradox)**。

* **悖论定义**：当人们试图将不同组的数据合并在一起进行分析时，在分组中都存在的某种趋势，在合并后的总体数据中可能会减弱、消失甚至逆转。
* **在您问题中的体现**：
    * **分组趋势**：在每一个独立的测试Fold（年份）这个“组”内，您的模型预测和真实收益之间都存在一个较强的正相关关系（平均IC高达 `0.2736`）。
    * **合并后趋势**：当您将所有这些“组”拼接在一起时，这个正相关关系被大幅削弱（拼接后IC降至 `0.0503`）。

**为什么会发生这个悖论？**
因为您正在处理的是典型的**非平稳（Non-stationary）**金融时间序列数据。这意味着数据的统计特性（例如均值、方差）会随着时间而改变。
* **“市场环境”（Market Regime）**是这个悖论中的**“混淆变量”（Confounding Variable）**。2014年的牛市和2022年的熊市，是两个性质完全不同的“总体”。
* 您的滚动回测方法，其设计的核心目的，恰恰就是为了**捕捉和评估**模型在这种非平稳性下的表现。

---

### 第二部分：数据结构分析 —— 为什么训练集与测试集表现不同？

现在，我们来解答最关键的问题：为什么训练集没有出现这种悖论，而测试集却表现得如此明显？答案在于您代码中构造这两类数据集的方式。

#### A. 训练集：一个“扩张与重叠”的稳定世界

您的代码通过`load_fold_data`函数，为每个Fold生成**扩张窗口（Expanding Window）**的训练集：
* `Fold 1 训练集` = `2009-2013`年
* `Fold 2 训练集` = `2009-2014`年
* `Fold 3 训练集` = `2009-2015`年

这种结构导致了以下三个必然结果：

1.  **数据层面 - 高度同质化**：后一个Fold与前一个Fold的训练数据有超过90%是完全相同的。这使得所有Fold的训练集在数据分布、市场规律上都非常相似。
2.  **模型层面 - 高度相似**: 由于输入的“教材”基本相同，为每个Fold训练出的XGBoost模型，其内部学到的规律和“认知”也是高度相似的。
3.  **预测层面 - 行为一致**: 当这些高度相似的模型，对它们各自的、高度重叠的训练集进行预测时（这是一个in-sample预测），它们输出的预测分，其**分布和含义（即我们讨论的“刻度尺”）**自然也是非常稳定和一致的。

**推论**：当您将这些来自**高度相似模型**、作用于**高度重叠数据**的预测结果拼接在一起时，它们之间几乎没有“排名倒挂”的冲突。因此，训练集的“拼接后IC”自然就约等于“平均IC”。

#### B. 测试集：一个“独立与未知”的真实考验

与训练集完全相反，不同Fold的**测试集是完全独立、不重叠的**：
* `Fold 1 测试集` = `2014`年
* `Fold 2 测试集` = `2015`年
* `Fold 3 测试集` = `2016`年

这种结构完美地模拟了真实世界，并导致了：

1.  **数据层面 - 高度异质化**: 每一个Fold的测试集都是一个独立的、全新的市场环境。
2.  **模型层面 - 接受真实考验**: 模型在面对每一个全新的年份时，都在接受一次真正的、对未知情况的考验。
3.  **预测层面 - “刻度尺”变化**: 模型为了适应不同的市场环境，其输出的预测分“刻度尺”会发生变化。在牛市中，一个`0.01`的预测分可能是个很差的信号；但在熊市中，`0.01`的预测分可能已经代表了最好的投资机会。

**推论**：拼接测试集的过程，就是将来自**不同市场环境**、由**不同“刻度尺”**度量出的预测结果，强行放在一个统一的排名体系下。这必然会因为市场规律的冲突而产生大量的“排名倒挂”，从而触发辛普森悖论。

---

### 第三部分：数学细节 —— “排名倒挂”如何摧毁IC？

让我们通过一个具体的例子，来观察这个“排名倒挂”在数学上是如何发生的。

**斯皮尔曼相关系数**的一个计算公式是：
$$\rho_s = 1 - \frac{6 \sum d_i^2}{n(n^2 - 1)}$$
其中，$d_i$ 是第 $i$ 个数据点的“预测排名”与“真实排名”之差。IC的大小，与**排名差异的平方和 $\sum d_i^2$** 成反比。

**情景模拟**：
* **Fold A (牛市)**: 股票B，模型预测分`0.015` (在该Fold中排名较低)，真实收益`2%`。
* **Fold B (熊市)**: 股票X，模型预测分`0.020` (在该Fold中排名很高)，真实收益`-1%`。

**拼接后进行全局排名时**：
1.  **预测排名**: 因为 `0.020 > 0.015`，所以 **X的预测排名高于B**。
2.  **真实排名**: 因为 `2% > -1%`，所以 **B的真实排名高于X**。

这就产生了一次**“排名倒挂”**。假设在全局排名中，X的预测排名是第1000名，真实排名是第1500名；而B的预测排名是第1010名，真实排名是第900名。
* 对于X, $d_X^2 = (1000 - 1500)^2 = 250000$
* 对于B, $d_B^2 = (1010 - 900)^2 = 12100$

**数学结论**：
仅仅是这两个数据点之间的“倒挂”，就给分子 $\sum d_i^2$ 贡献了巨大的数值。当成百上千次这样的、由市场环境切换导致的系统性“倒挂”累积起来，就会使 $\sum d_i^2$ 变得巨大。

**关于分母**：分母以$n^3$的速度增长。但问题的关键是，在一个充满“排名倒挂”的、近似于结构性随机的系统中，分子 $\sum d_i^2$ 的增长速度**也开始向 $n^3$ 级别靠拢**。正是因为分子和分母以相似的幂次增长，它们的比率才不会趋向于0，而是趋向于一个显著的非零常数，从而将Rank IC从完美的+1.0，拉向了更接近0的数值。



In [107]:
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

# ================================
# 配置参数
# ================================
DATA_PATH = 'QQQ_Rolling_Validation/QQQ_rolling_validation_dataset.xlsx'
OUTPUT_DIR = 'QQQ_XGBoost_Stable_Features_Fixed_Params'
TARGET_NAME = '目标_未来10d回报率'

# 稳定特征文件路径
STABLE_FEATURES_FILE = 'QQQ_XGBoost_IC_Analysis/stable_features.txt'

# 固定的XGBoost参数
FIXED_XGBOOST_PARAMS = {
    'n_estimators': 150,
    'learning_rate': 0.05,
    'max_depth': 3,
    'subsample': 0.9,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.5,
    'reg_lambda': 1.5,    
    'min_child_weight': 1,
    'gamma': 0,
    'objective': 'reg:squarederror',
    'random_state': 42,
    'n_jobs': -1
}

# 策略参数
SHAP_SAMPLE_SIZE = 500

# ================================
# 核心函数
# ================================

def set_chinese_font():
    """设置中文字体"""
    try:
        plt.rcParams['font.sans-serif'] = ['SimHei']
        plt.rcParams['axes.unicode_minus'] = False
    except:
        pass

def load_stable_features(file_path=STABLE_FEATURES_FILE):
    """加载稳定特征列表"""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"稳定特征文件不存在: {file_path}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        features = [line.strip() for line in f if line.strip()]
    
    print(f"从 {file_path} 加载了 {len(features)} 个稳定特征")
    return features

def get_available_folds(data_path):
    """获取所有可用的fold编号"""
    excel_file = pd.ExcelFile(data_path)
    fold_sheets = [sheet for sheet in excel_file.sheet_names if sheet.startswith('Fold') and sheet.endswith('_训练集')]
    fold_numbers = [int(sheet.split('Fold')[1].split('_')[0]) for sheet in fold_sheets]
    return sorted(fold_numbers)

def load_fold_data(data_path, fold_number, target_name, stable_features):
    """加载fold数据，只使用稳定特征"""
    train_sheet = f'Fold{fold_number:02d}_训练集'
    train_df = pd.read_excel(data_path, sheet_name=train_sheet, index_col=0, parse_dates=True)
    train_df = train_df.dropna()
    
    test_sheet = f'Fold{fold_number:02d}_测试集'
    test_df = pd.read_excel(data_path, sheet_name=test_sheet, index_col=0, parse_dates=True)
    test_df = test_df.dropna()
    
    # 检查稳定特征是否存在于数据中
    available_features = [f for f in stable_features if f in train_df.columns]
    missing_features = [f for f in stable_features if f not in train_df.columns]
    
    if missing_features:
        print(f"  警告: {len(missing_features)} 个稳定特征在数据中不存在")
        if len(missing_features) <= 5:
            print(f"  缺失特征: {missing_features}")
    
    print(f"  使用 {len(available_features)} 个可用稳定特征")
    
    # 使用可用的稳定特征
    X_train = train_df[available_features]
    y_train = train_df[target_name]
    X_test = test_df[available_features]
    y_test = test_df[target_name]
    
    return X_train, y_train, X_test, y_test, available_features

def calculate_ic(predictions, targets, method='spearman'):
    """计算IC值"""
    if method == 'pearson':
        ic, p_value = stats.pearsonr(predictions, targets)
    elif method == 'spearman':
        ic, p_value = stats.spearmanr(predictions, targets)
    return ic, p_value

def rank_normalize_predictions(predictions):
    """
    对预测值进行排序归一化处理
    将预测值转换为百分位排名 (0-1之间)
    """
    # 计算排名 (rank)，使用 'average' 方法处理相同值
    ranks = stats.rankdata(predictions, method='average')
    # 转换为百分位排名 (0-1之间)
    percentile_ranks = (ranks - 1) / (len(ranks) - 1) if len(ranks) > 1 else np.zeros_like(ranks)
    return percentile_ranks

def train_fixed_model(X_train, y_train, X_test, y_test, fixed_params):
    """使用固定参数训练模型并计算IC"""
    print("  使用固定参数训练模型...")
    print(f"  固定参数: {fixed_params}")
    
    final_model = xgb.XGBRegressor(**fixed_params)
    final_model.fit(X_train, y_train)
    
    y_train_pred = final_model.predict(X_train)
    y_test_pred = final_model.predict(X_test)
    
    # 转换为Series并保持原索引
    y_train_pred = pd.Series(y_train_pred, index=y_train.index)
    y_test_pred = pd.Series(y_test_pred, index=y_test.index)
    
    # 对每个Fold内部的预测值进行排序归一化
    y_train_pred_normalized = pd.Series(
        rank_normalize_predictions(y_train_pred.values), 
        index=y_train.index
    )
    y_test_pred_normalized = pd.Series(
        rank_normalize_predictions(y_test_pred.values), 
        index=y_test.index
    )
    
    # 计算原始预测值的IC
    train_ic, train_ic_pval = calculate_ic(y_train_pred.values, y_train.values, method='spearman')
    test_ic, test_ic_pval = calculate_ic(y_test_pred.values, y_test.values, method='spearman')
    
    # 计算归一化预测值的IC
    train_ic_norm, train_ic_pval_norm = calculate_ic(y_train_pred_normalized.values, y_train.values, method='spearman')
    test_ic_norm, test_ic_pval_norm = calculate_ic(y_test_pred_normalized.values, y_test.values, method='spearman')
    
    print(f"    原始预测值 - 训练集Spearman IC: {train_ic:.4f} (p={train_ic_pval:.4f})")
    print(f"    原始预测值 - 测试集Spearman IC: {test_ic:.4f} (p={test_ic_pval:.4f})")
    print(f"    归一化预测值 - 训练集Spearman IC: {train_ic_norm:.4f} (p={train_ic_pval_norm:.4f})")
    print(f"    归一化预测值 - 测试集Spearman IC: {test_ic_norm:.4f} (p={test_ic_pval_norm:.4f})")
    
    return (final_model, y_train_pred, y_test_pred, train_ic, test_ic, train_ic_pval, test_ic_pval,
            y_train_pred_normalized, y_test_pred_normalized, train_ic_norm, test_ic_norm, 
            train_ic_pval_norm, test_ic_pval_norm)

def create_prediction_distribution_plot(y_train_pred, y_test_pred, y_train, y_test, 
                                      y_train_pred_norm, y_test_pred_norm, fold_number, fold_dir):
    """为每个fold创建预测值分布图（包含归一化结果）"""
    set_chinese_font()
    
    fig, axes = plt.subplots(3, 2, figsize=(14, 15))
    
    # 第一行：原始预测值分布
    axes[0, 0].hist(y_train_pred, bins=30, alpha=0.7, color='blue', edgecolor='black', label='预测值')
    axes[0, 0].axvline(y_train_pred.mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {y_train_pred.mean():.4f}')
    axes[0, 0].set_title(f'Fold {fold_number:02d} - 训练集原始预测值分布')
    axes[0, 0].set_xlabel('预测值')
    axes[0, 0].set_ylabel('频数')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].hist(y_test_pred, bins=30, alpha=0.7, color='green', edgecolor='black', label='预测值')
    axes[0, 1].axvline(y_test_pred.mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {y_test_pred.mean():.4f}')
    axes[0, 1].set_title(f'Fold {fold_number:02d} - 测试集原始预测值分布')
    axes[0, 1].set_xlabel('预测值')
    axes[0, 1].set_ylabel('频数')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 第二行：归一化预测值分布
    axes[1, 0].hist(y_train_pred_norm, bins=30, alpha=0.7, color='purple', edgecolor='black', label='归一化预测值')
    axes[1, 0].axvline(y_train_pred_norm.mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {y_train_pred_norm.mean():.4f}')
    axes[1, 0].set_title(f'Fold {fold_number:02d} - 训练集归一化预测值分布')
    axes[1, 0].set_xlabel('归一化预测值 (百分位排名)')
    axes[1, 0].set_ylabel('频数')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].hist(y_test_pred_norm, bins=30, alpha=0.7, color='orange', edgecolor='black', label='归一化预测值')
    axes[1, 1].axvline(y_test_pred_norm.mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {y_test_pred_norm.mean():.4f}')
    axes[1, 1].set_title(f'Fold {fold_number:02d} - 测试集归一化预测值分布')
    axes[1, 1].set_xlabel('归一化预测值 (百分位排名)')
    axes[1, 1].set_ylabel('频数')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # 第三行：预测值vs真实值对比分布
    axes[2, 0].hist(y_train, bins=30, alpha=0.5, color='blue', edgecolor='black', label='真实值')
    axes[2, 0].hist(y_train_pred, bins=30, alpha=0.5, color='red', edgecolor='black', label='原始预测值')
    axes[2, 0].set_title(f'Fold {fold_number:02d} - 训练集预测vs真实分布')
    axes[2, 0].set_xlabel('值')
    axes[2, 0].set_ylabel('频数')
    axes[2, 0].legend()
    axes[2, 0].grid(True, alpha=0.3)
    
    axes[2, 1].hist(y_test, bins=30, alpha=0.5, color='blue', edgecolor='black', label='真实值')
    axes[2, 1].hist(y_test_pred, bins=30, alpha=0.5, color='red', edgecolor='black', label='原始预测值')
    axes[2, 1].set_title(f'Fold {fold_number:02d} - 测试集预测vs真实分布')
    axes[2, 1].set_xlabel('值')
    axes[2, 1].set_ylabel('频数')
    axes[2, 1].legend()
    axes[2, 1].grid(True, alpha=0.3)
    
    fig.suptitle(f'Fold {fold_number:02d} 预测值分布分析（含归一化）', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(fold_dir, f'prediction_distribution.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  已生成Fold {fold_number:02d}的预测值分布图")

def create_fold_scatter_plot(y_train_pred, y_test_pred, y_train, y_test, 
                           y_train_pred_norm, y_test_pred_norm, fold_number, fold_dir):
    """为每个fold创建散点图（包含归一化结果）"""
    # 计算IC
    train_ic, train_ic_pval = calculate_ic(y_train_pred.values, y_train.values, method='spearman')
    test_ic, test_ic_pval = calculate_ic(y_test_pred.values, y_test.values, method='spearman')
    train_ic_norm, train_ic_pval_norm = calculate_ic(y_train_pred_norm.values, y_train.values, method='spearman')
    test_ic_norm, test_ic_pval_norm = calculate_ic(y_test_pred_norm.values, y_test.values, method='spearman')
    
    # 创建散点图
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            f'训练集原始预测vs实际 (IC: {train_ic:.4f})',
            f'测试集原始预测vs实际 (IC: {test_ic:.4f})',
            f'训练集归一化预测vs实际 (IC: {train_ic_norm:.4f})',
            f'测试集归一化预测vs实际 (IC: {test_ic_norm:.4f})'
        ]
    )
    
    # 原始预测值散点图
    fig.add_trace(go.Scatter(
        x=y_train.values,
        y=y_train_pred.values,
        mode='markers',
        name='训练集原始',
        marker=dict(color='blue', opacity=0.6, size=3),
        showlegend=False
    ), row=1, col=1)
    
    fig.add_trace(go.Scatter(
        x=y_test.values,
        y=y_test_pred.values,
        mode='markers',
        name='测试集原始',
        marker=dict(color='red', opacity=0.6, size=3),
        showlegend=False
    ), row=1, col=2)
    
    # 归一化预测值散点图
    fig.add_trace(go.Scatter(
        x=y_train.values,
        y=y_train_pred_norm.values,
        mode='markers',
        name='训练集归一化',
        marker=dict(color='purple', opacity=0.6, size=3),
        showlegend=False
    ), row=2, col=1)
    
    fig.add_trace(go.Scatter(
        x=y_test.values,
        y=y_test_pred_norm.values,
        mode='markers',
        name='测试集归一化',
        marker=dict(color='orange', opacity=0.6, size=3),
        showlegend=False
    ), row=2, col=2)
    
    fig.update_layout(
        title=f'Fold {fold_number:02d} 预测效果分析（原始 vs 归一化）',
        height=800
    )
    
    fig.update_xaxes(title_text="实际值")
    fig.update_yaxes(title_text="预测值", row=1)
    fig.update_yaxes(title_text="归一化预测值", row=2)
    
    # 保存图片
    scatter_file = os.path.join(fold_dir, f'fold_{fold_number:02d}_scatter.html')
    fig.write_html(scatter_file)
    
    print(f"  已生成Fold {fold_number:02d}的散点图: {scatter_file}")

def process_single_fold(data_path, fold_number, target_name, stable_features, fixed_params, output_dir):
    """处理单个fold"""
    print(f"\n处理 Fold {fold_number}")
    print("=" * 50)
    
    # 加载数据，使用稳定特征
    X_train, y_train, X_test, y_test, used_features = load_fold_data(data_path, fold_number, target_name, stable_features)
    
    print(f"数据规模: 训练集 {X_train.shape}, 测试集 {X_test.shape}")
    print(f"训练集时间范围: {X_train.index[0].strftime('%Y-%m-%d')} 至 {X_train.index[-1].strftime('%Y-%m-%d')}")
    print(f"测试集时间范围: {X_test.index[0].strftime('%Y-%m-%d')} 至 {X_test.index[-1].strftime('%Y-%m-%d')}")
    
    # 训练固定参数模型
    (final_model, y_train_pred, y_test_pred, train_ic, test_ic, train_ic_pval, test_ic_pval,
     y_train_pred_normalized, y_test_pred_normalized, train_ic_norm, test_ic_norm, 
     train_ic_pval_norm, test_ic_pval_norm) = train_fixed_model(X_train, y_train, X_test, y_test, fixed_params)
    
    # 保存结果
    fold_dir = os.path.join(output_dir, f'Fold{fold_number:02d}')
    os.makedirs(fold_dir, exist_ok=True)
    
    # 保存预测结果（包含归一化结果）
    results_df = pd.DataFrame({
        'Date': y_train.index.tolist() + y_test.index.tolist(),
        'Type': ['Train'] * len(y_train) + ['Test'] * len(y_test),
        'Prediction': y_train_pred.tolist() + y_test_pred.tolist(),
        'Prediction_Normalized': y_train_pred_normalized.tolist() + y_test_pred_normalized.tolist(),
        'Actual': y_train.tolist() + y_test.tolist()
    })
    results_df.to_csv(os.path.join(fold_dir, 'predictions.csv'), index=False)
    
    # 保存使用的特征列表
    pd.DataFrame({'Feature': used_features}).to_csv(os.path.join(fold_dir, 'used_features.csv'), index=False)
    
    # 保存固定的超参数
    with open(os.path.join(fold_dir, 'fixed_hyperparams.json'), 'w', encoding='utf-8') as f:
        json.dump(fixed_params, f, indent=2, ensure_ascii=False)
    
    # 创建预测值分布图
    create_prediction_distribution_plot(y_train_pred, y_test_pred, y_train, y_test, 
                                      y_train_pred_normalized, y_test_pred_normalized, fold_number, fold_dir)
    
    # 创建散点图
    create_fold_scatter_plot(y_train_pred, y_test_pred, y_train, y_test, 
                           y_train_pred_normalized, y_test_pred_normalized, fold_number, fold_dir)
    
    return {
        'fold': fold_number,
        'train_ic': train_ic,
        'test_ic': test_ic,
        'train_ic_pval': train_ic_pval,
        'test_ic_pval': test_ic_pval,
        'train_ic_norm': train_ic_norm,
        'test_ic_norm': test_ic_norm,
        'train_ic_pval_norm': train_ic_pval_norm,
        'test_ic_pval_norm': test_ic_pval_norm,
        'used_features': used_features,
        'fixed_params': fixed_params,
        'test_predictions': y_test_pred,
        'test_actual': y_test,
        'train_predictions': y_train_pred,
        'train_actual': y_train,
        'test_predictions_normalized': y_test_pred_normalized,
        'train_predictions_normalized': y_train_pred_normalized
    }

def export_combined_test_predictions(all_results, output_dir):
    """导出所有fold测试集拼起来的预测值和真实值到Excel（包含归一化结果）"""
    print(f"\n导出合并测试集预测结果到Excel")
    print("=" * 50)
    
    combined_data = []
    combined_data_normalized = []
    
    for result in all_results:
        fold_number = result['fold']
        test_predictions = result['test_predictions']
        test_actual = result['test_actual']
        test_predictions_normalized = result['test_predictions_normalized']
        
        # 原始预测值数据
        fold_df = pd.DataFrame({
            'Date': test_predictions.index,
            'Fold': fold_number,
            'Prediction': test_predictions.values,
            'Actual': test_actual.values
        })
        combined_data.append(fold_df)
        
        # 归一化预测值数据
        fold_df_norm = pd.DataFrame({
            'Date': test_predictions_normalized.index,
            'Fold': fold_number,
            'Prediction_Normalized': test_predictions_normalized.values,
            'Actual': test_actual.values
        })
        combined_data_normalized.append(fold_df_norm)
    
    # 合并原始预测值
    combined_test_df = pd.concat(combined_data, ignore_index=True)
    combined_test_df = combined_test_df.sort_values('Date').reset_index(drop=True)
    
    # 合并归一化预测值
    combined_test_df_norm = pd.concat(combined_data_normalized, ignore_index=True)
    combined_test_df_norm = combined_test_df_norm.sort_values('Date').reset_index(drop=True)
    
    # 计算误差指标
    combined_test_df['Error'] = combined_test_df['Prediction'] - combined_test_df['Actual']
    combined_test_df['Abs_Error'] = np.abs(combined_test_df['Error'])
    combined_test_df['Squared_Error'] = combined_test_df['Error'] ** 2
    
    # 计算原始IC
    overall_ic, overall_pval = calculate_ic(
        combined_test_df['Prediction'].values, 
        combined_test_df['Actual'].values, 
        method='spearman'
    )
    
    # 计算归一化IC
    overall_ic_norm, overall_pval_norm = calculate_ic(
        combined_test_df_norm['Prediction_Normalized'].values, 
        combined_test_df_norm['Actual'].values, 
        method='spearman'
    )
    
    # 统计摘要
    summary_stats = pd.DataFrame({
        'Metric': ['Sample Count', 'Original IC', 'Original IC P-value', 'Normalized IC', 'Normalized IC P-value',
                   'Mean Prediction', 'Mean Actual', 'Std Prediction', 'Std Actual', 'MAE', 'MSE', 'RMSE'],
        'Value': [
            len(combined_test_df),
            overall_ic,
            overall_pval,
            overall_ic_norm,
            overall_pval_norm,
            combined_test_df['Prediction'].mean(),
            combined_test_df['Actual'].mean(),
            combined_test_df['Prediction'].std(),
            combined_test_df['Actual'].std(),
            combined_test_df['Abs_Error'].mean(),
            combined_test_df['Squared_Error'].mean(),
            np.sqrt(combined_test_df['Squared_Error'].mean())
        ]
    })
    
    # 合并原始和归一化数据
    combined_final_df = combined_test_df.copy()
    combined_final_df['Prediction_Normalized'] = combined_test_df_norm['Prediction_Normalized'].values
    
    excel_path = os.path.join(output_dir, 'combined_test_predictions.xlsx')
    
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        combined_final_df.to_excel(writer, sheet_name='测试集预测数据', index=False)
        summary_stats.to_excel(writer, sheet_name='统计摘要', index=False)
        
        fold_summary = combined_test_df.groupby('Fold').agg({
            'Prediction': ['mean', 'std', 'min', 'max'],
            'Actual': ['mean', 'std', 'min', 'max'],
            'Error': ['mean', 'std'],
            'Abs_Error': 'mean'
        }).round(6)
        fold_summary.to_excel(writer, sheet_name='各Fold统计')
    
    print(f"  已导出合并测试集数据到: {excel_path}")
    print(f"  总样本数: {len(combined_test_df)}")
    print(f"  时间范围: {combined_test_df['Date'].min().strftime('%Y-%m-%d')} 至 {combined_test_df['Date'].max().strftime('%Y-%m-%d')}")
    print(f"  整体测试集原始IC: {overall_ic:.4f} (p={overall_pval:.4f})")
    print(f"  整体测试集归一化IC: {overall_ic_norm:.4f} (p={overall_pval_norm:.4f})")
    
    return combined_final_df, summary_stats

def create_summary_analysis(all_results, output_dir):
    """创建汇总分析（包含归一化结果）"""
    print(f"\n创建汇总分析")
    print("=" * 50)
    
    summary_data = []
    for result in all_results:
        summary_data.append({
            'Fold': result['fold'],
            'Train_IC': result['train_ic'],
            'Test_IC': result['test_ic'],
            'Train_IC_PValue': result['train_ic_pval'],
            'Test_IC_PValue': result['test_ic_pval'],
            'Train_IC_Norm': result['train_ic_norm'],
            'Test_IC_Norm': result['test_ic_norm'],
            'Train_IC_PValue_Norm': result['train_ic_pval_norm'],
            'Test_IC_PValue_Norm': result['test_ic_pval_norm'],
            'IC_Significant': result['test_ic_pval'] < 0.05,
            'IC_Significant_Norm': result['test_ic_pval_norm'] < 0.05
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    # 计算统计指标
    print(f"各Fold IC结果:")
    for _, row in summary_df.iterrows():
        significance = "***" if row['Test_IC_PValue'] < 0.01 else "**" if row['Test_IC_PValue'] < 0.05 else "*" if row['Test_IC_PValue'] < 0.1 else ""
        significance_norm = "***" if row['Test_IC_PValue_Norm'] < 0.01 else "**" if row['Test_IC_PValue_Norm'] < 0.05 else "*" if row['Test_IC_PValue_Norm'] < 0.1 else ""
        print(f"  Fold {int(row['Fold']):02d}: 原始IC={row['Test_IC']:.4f}{significance} (p={row['Test_IC_PValue']:.4f}) | 归一化IC={row['Test_IC_Norm']:.4f}{significance_norm} (p={row['Test_IC_PValue_Norm']:.4f})")
    
    print(f"\n整体IC统计:")
    print(f"平均IC (训练集原始): {summary_df['Train_IC'].mean():.4f} ± {summary_df['Train_IC'].std():.4f}")
    print(f"平均IC (测试集原始): {summary_df['Test_IC'].mean():.4f} ± {summary_df['Test_IC'].std():.4f}")
    print(f"平均IC (训练集归一化): {summary_df['Train_IC_Norm'].mean():.4f} ± {summary_df['Train_IC_Norm'].std():.4f}")
    print(f"平均IC (测试集归一化): {summary_df['Test_IC_Norm'].mean():.4f} ± {summary_df['Test_IC_Norm'].std():.4f}")
    print(f"原始IC显著性(p<0.05): {summary_df['IC_Significant'].sum()}/{len(summary_df)} ({summary_df['IC_Significant'].mean():.1%})")
    print(f"归一化IC显著性(p<0.05): {summary_df['IC_Significant_Norm'].sum()}/{len(summary_df)} ({summary_df['IC_Significant_Norm'].mean():.1%})")
    print(f"原始IC胜率: {(summary_df['Test_IC'] > 0).sum()}/{len(summary_df)} ({(summary_df['Test_IC'] > 0).mean():.1%})")
    print(f"归一化IC胜率: {(summary_df['Test_IC_Norm'] > 0).sum()}/{len(summary_df)} ({(summary_df['Test_IC_Norm'] > 0).mean():.1%})")
    
    # 创建全数据集IC分析
    overall_results = create_overall_ic_analysis(all_results, output_dir)
    
    # 导出合并测试集预测结果
    combined_test_df, summary_stats = export_combined_test_predictions(all_results, output_dir)
    
    # 保存结果
    with pd.ExcelWriter(os.path.join(output_dir, 'ic_analysis_results.xlsx')) as writer:
        summary_df.to_excel(writer, sheet_name='各Fold_IC结果', index=False)
    
    # 保存固定参数配置
    save_fixed_hyperparams(all_results, output_dir)
    
    return summary_df

def create_overall_ic_analysis(all_results, output_dir):
    """创建全数据集的IC分析（包含归一化结果）"""
    print(f"\n创建全数据集IC分析")
    print("=" * 50)
    
    # 拼接所有测试集的预测和实际值
    all_test_predictions = []
    all_test_actual = []
    all_train_predictions = []
    all_train_actual = []
    all_test_predictions_norm = []
    all_train_predictions_norm = []
    all_dates_test = []
    all_dates_train = []
    
    for result in all_results:
        all_test_predictions.extend(result['test_predictions'].tolist())
        all_test_actual.extend(result['test_actual'].tolist())
        all_train_predictions.extend(result['train_predictions'].tolist())
        all_train_actual.extend(result['train_actual'].tolist())
        all_test_predictions_norm.extend(result['test_predictions_normalized'].tolist())
        all_train_predictions_norm.extend(result['train_predictions_normalized'].tolist())
        all_dates_test.extend(result['test_predictions'].index.tolist())
        all_dates_train.extend(result['train_predictions'].index.tolist())
    
    # 计算全数据集IC - 原始预测值
    overall_test_ic, overall_test_ic_pval = calculate_ic(all_test_predictions, all_test_actual, method='spearman')
    overall_train_ic, overall_train_ic_pval = calculate_ic(all_train_predictions, all_train_actual, method='spearman')
    
    # 计算全数据集IC - 归一化预测值
    overall_test_ic_norm, overall_test_ic_pval_norm = calculate_ic(all_test_predictions_norm, all_test_actual, method='spearman')
    overall_train_ic_norm, overall_train_ic_pval_norm = calculate_ic(all_train_predictions_norm, all_train_actual, method='spearman')
    
    # 输出结果
    print(f"全数据集IC分析:")
    print(f"训练集合并原始IC: {overall_train_ic:.4f} (p={overall_train_ic_pval:.4f})")
    print(f"测试集合并原始IC: {overall_test_ic:.4f} (p={overall_test_ic_pval:.4f})")
    print(f"训练集合并归一化IC: {overall_train_ic_norm:.4f} (p={overall_train_ic_pval_norm:.4f})")
    print(f"测试集合并归一化IC: {overall_test_ic_norm:.4f} (p={overall_test_ic_pval_norm:.4f})")
    print(f"测试集数据点数: {len(all_test_predictions)}")
    print(f"训练集数据点数: {len(all_train_predictions)}")
    
    if all_dates_test:
        print(f"测试集时间跨度: {min(all_dates_test).strftime('%Y-%m-%d')} 至 {max(all_dates_test).strftime('%Y-%m-%d')}")
    if all_dates_train:
        print(f"训练集时间跨度: {min(all_dates_train).strftime('%Y-%m-%d')} 至 {max(all_dates_train).strftime('%Y-%m-%d')}")
    
    # 保存全数据集结果
    overall_summary = pd.DataFrame({
        'Dataset': ['训练集合并原始', '测试集合并原始', '训练集合并归一化', '测试集合并归一化'],
        'IC': [overall_train_ic, overall_test_ic, overall_train_ic_norm, overall_test_ic_norm],
        'P_Value': [overall_train_ic_pval, overall_test_ic_pval, overall_train_ic_pval_norm, overall_test_ic_pval_norm],
        'Significant': [overall_train_ic_pval < 0.05, overall_test_ic_pval < 0.05, 
                       overall_train_ic_pval_norm < 0.05, overall_test_ic_pval_norm < 0.05],
        'Data_Points': [len(all_train_predictions), len(all_test_predictions), 
                       len(all_train_predictions), len(all_test_predictions)]
    })
    
    # 创建散点图
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            f'训练集原始预测vs实际 (IC: {overall_train_ic:.4f})',
            f'测试集原始预测vs实际 (IC: {overall_test_ic:.4f})',
            f'训练集归一化预测vs实际 (IC: {overall_train_ic_norm:.4f})',
            f'测试集归一化预测vs实际 (IC: {overall_test_ic_norm:.4f})'
        ]
    )
    
    # 原始预测值散点图
    fig.add_trace(go.Scatter(
        x=all_train_actual,
        y=all_train_predictions,
        mode='markers',
        name='训练集原始',
        marker=dict(color='blue', opacity=0.6, size=3),
        showlegend=False
    ), row=1, col=1)
    
    fig.add_trace(go.Scatter(
        x=all_test_actual,
        y=all_test_predictions,
        mode='markers',
        name='测试集原始',
        marker=dict(color='red', opacity=0.6, size=3),
        showlegend=False
    ), row=1, col=2)
    
    # 归一化预测值散点图
    fig.add_trace(go.Scatter(
        x=all_train_actual,
        y=all_train_predictions_norm,
        mode='markers',
        name='训练集归一化',
        marker=dict(color='purple', opacity=0.6, size=3),
        showlegend=False
    ), row=2, col=1)
    
    fig.add_trace(go.Scatter(
        x=all_test_actual,
        y=all_test_predictions_norm,
        mode='markers',
        name='测试集归一化',
        marker=dict(color='orange', opacity=0.6, size=3),
        showlegend=False
    ), row=2, col=2)
    
    fig.update_layout(
        title='全数据集预测效果分析（原始 vs 归一化）',
        height=800
    )
    
    fig.update_xaxes(title_text="实际值")
    fig.update_yaxes(title_text="原始预测值", row=1)
    fig.update_yaxes(title_text="归一化预测值", row=2)
    
    # 保存结果
    with pd.ExcelWriter(os.path.join(output_dir, 'overall_ic_analysis.xlsx')) as writer:
        overall_summary.to_excel(writer, sheet_name='全数据集IC分析', index=False)
        
        # 保存详细数据
        overall_data = pd.DataFrame({
            'Date': all_dates_train + all_dates_test,
            'Type': ['Train'] * len(all_dates_train) + ['Test'] * len(all_dates_test),
            'Prediction': all_train_predictions + all_test_predictions,
            'Prediction_Normalized': all_train_predictions_norm + all_test_predictions_norm,
            'Actual': all_train_actual + all_test_actual
        })
        overall_data.to_excel(writer, sheet_name='全数据集详细数据', index=False)
    
    fig.write_html(os.path.join(output_dir, 'overall_ic_scatter.html'))
    
    return {
        'overall_train_ic': overall_train_ic,
        'overall_test_ic': overall_test_ic,
        'overall_train_ic_norm': overall_train_ic_norm,
        'overall_test_ic_norm': overall_test_ic_norm,
        'summary': overall_summary
    }

def save_fixed_hyperparams(all_results, output_dir):
    """保存固定的超参数配置"""
    print(f"\n保存固定参数配置")
    print("=" * 50)
    
    # 保存固定参数
    fixed_params_json = os.path.join(output_dir, 'fixed_hyperparams.json')
    with open(fixed_params_json, 'w', encoding='utf-8') as f:
        json.dump(FIXED_XGBOOST_PARAMS, f, indent=2, ensure_ascii=False)
    
    print(f"已保存固定参数配置到: {fixed_params_json}")
    print(f"使用的固定参数: {FIXED_XGBOOST_PARAMS}")
    
    return FIXED_XGBOOST_PARAMS

def create_ic_visualization(summary_df, output_dir):
    """创建IC可视化（包含归一化结果）"""
    set_chinese_font()
    fig, axes = plt.subplots(3, 2, figsize=(15, 15))
    
    # 第一行：原始IC分析
    # IC分布直方图
    axes[0, 0].hist(summary_df['Test_IC'], bins=10, alpha=0.7, color='blue', edgecolor='black')
    axes[0, 0].axvline(summary_df['Test_IC'].mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {summary_df["Test_IC"].mean():.3f}')
    axes[0, 0].axvline(0, color='black', linestyle='-', alpha=0.5)
    axes[0, 0].set_title('测试集原始IC分布')
    axes[0, 0].set_xlabel('IC值')
    axes[0, 0].set_ylabel('频数')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 各Fold IC对比
    x_pos = range(len(summary_df))
    colors = ['green' if ic > 0 else 'red' for ic in summary_df['Test_IC']]
    bars = axes[0, 1].bar(x_pos, summary_df['Test_IC'], color=colors, alpha=0.7)
    axes[0, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
    axes[0, 1].axhline(y=summary_df['Test_IC'].mean(), color='blue', linestyle='--', linewidth=2, label=f'平均IC: {summary_df["Test_IC"].mean():.3f}')
    axes[0, 1].set_title('各Fold测试集原始IC')
    axes[0, 1].set_xlabel('Fold')
    axes[0, 1].set_ylabel('IC值')
    axes[0, 1].set_xticks(x_pos)
    axes[0, 1].set_xticklabels([f'F{int(fold):02d}' for fold in summary_df['Fold']])
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 第二行：归一化IC分析
    # IC分布直方图
    axes[1, 0].hist(summary_df['Test_IC_Norm'], bins=10, alpha=0.7, color='purple', edgecolor='black')
    axes[1, 0].axvline(summary_df['Test_IC_Norm'].mean(), color='red', linestyle='--', linewidth=2, label=f'均值: {summary_df["Test_IC_Norm"].mean():.3f}')
    axes[1, 0].axvline(0, color='black', linestyle='-', alpha=0.5)
    axes[1, 0].set_title('测试集归一化IC分布')
    axes[1, 0].set_xlabel('IC值')
    axes[1, 0].set_ylabel('频数')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 各Fold IC对比
    colors_norm = ['green' if ic > 0 else 'red' for ic in summary_df['Test_IC_Norm']]
    bars_norm = axes[1, 1].bar(x_pos, summary_df['Test_IC_Norm'], color=colors_norm, alpha=0.7)
    axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
    axes[1, 1].axhline(y=summary_df['Test_IC_Norm'].mean(), color='blue', linestyle='--', linewidth=2, label=f'平均IC: {summary_df["Test_IC_Norm"].mean():.3f}')
    axes[1, 1].set_title('各Fold测试集归一化IC')
    axes[1, 1].set_xlabel('Fold')
    axes[1, 1].set_ylabel('IC值')
    axes[1, 1].set_xticks(x_pos)
    axes[1, 1].set_xticklabels([f'F{int(fold):02d}' for fold in summary_df['Fold']])
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # 第三行：比较分析
    # 训练vs测试IC对比 (原始)
    axes[2, 0].scatter(summary_df['Train_IC'], summary_df['Test_IC'], alpha=0.7, s=60, color='blue', label='原始IC')
    axes[2, 0].plot([-0.3, 0.3], [-0.3, 0.3], 'r--', alpha=0.5, label='y=x')
    axes[2, 0].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    axes[2, 0].axvline(x=0, color='black', linestyle='-', alpha=0.3)
    axes[2, 0].set_xlabel('训练集原始IC')
    axes[2, 0].set_ylabel('测试集原始IC')
    axes[2, 0].set_title('训练集vs测试集原始IC')
    axes[2, 0].legend()
    axes[2, 0].grid(True, alpha=0.3)
    
    # 原始vs归一化IC对比
    axes[2, 1].scatter(summary_df['Test_IC'], summary_df['Test_IC_Norm'], alpha=0.7, s=60, color='orange')
    axes[2, 1].plot([-0.3, 0.3], [-0.3, 0.3], 'r--', alpha=0.5, label='y=x')
    axes[2, 1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    axes[2, 1].axvline(x=0, color='black', linestyle='-', alpha=0.3)
    axes[2, 1].set_xlabel('测试集原始IC')
    axes[2, 1].set_ylabel('测试集归一化IC')
    axes[2, 1].set_title('原始IC vs 归一化IC')
    axes[2, 1].legend()
    axes[2, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'ic_analysis_summary.png'), dpi=300, bbox_inches='tight')
    plt.close()

def main():
    """主函数"""
    if not os.path.exists(DATA_PATH):
        print(f"错误：数据文件不存在 {DATA_PATH}")
        return
    
    # 加载稳定特征
    try:
        stable_features = load_stable_features(STABLE_FEATURES_FILE)
    except FileNotFoundError as e:
        print(f"错误: {e}")
        print("请先运行原始分析代码生成稳定特征文件")
        return
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    available_folds = get_available_folds(DATA_PATH)
    
    print(f"XGBoost 稳定特征固定参数训练（含排序归一化）")
    print(f"发现 {len(available_folds)} 个fold")
    print(f"目标变量: {TARGET_NAME}")
    print(f"使用稳定特征数: {len(stable_features)}")
    print(f"IC计算方法: Spearman相关系数")
    print(f"策略: 使用稳定特征 + 固定超参数 + 排序归一化")
    print(f"排序归一化: 将每个Fold内预测值转换为百分位排名")
    print(f"固定参数:")
    for param, value in FIXED_XGBOOST_PARAMS.items():
        if param not in ['objective', 'random_state', 'n_jobs']:
            print(f"  {param}: {value}")
    
    all_results = []
    for fold_number in available_folds:
        try:
            result = process_single_fold(DATA_PATH, fold_number, TARGET_NAME, stable_features, FIXED_XGBOOST_PARAMS, OUTPUT_DIR)
            if result:
                all_results.append(result)
        except Exception as e:
            print(f"Fold {fold_number} 处理失败: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    if not all_results:
        print("错误：没有成功处理任何fold")
        return
    
    # 创建汇总分析
    summary_df = create_summary_analysis(all_results, OUTPUT_DIR)
    
    # 创建可视化
    create_ic_visualization(summary_df, OUTPUT_DIR)
    
    print(f"\n分析完成！")
    print(f"成功处理 {len(all_results)} 个fold")
    print(f"结果保存至: {OUTPUT_DIR}")
    
    # 最终总结
    print(f"\n" + "="*60)
    print(f"最终IC总结:")
    print(f"="*60)
    print(f"各Fold平均测试IC (原始): {summary_df['Test_IC'].mean():.4f} ± {summary_df['Test_IC'].std():.4f}")
    print(f"各Fold平均测试IC (归一化): {summary_df['Test_IC_Norm'].mean():.4f} ± {summary_df['Test_IC_Norm'].std():.4f}")
    print(f"原始IC胜率: {(summary_df['Test_IC'] > 0).mean():.1%}")
    print(f"归一化IC胜率: {(summary_df['Test_IC_Norm'] > 0).mean():.1%}")
    print(f"原始显著IC(p<0.05)占比: {summary_df['IC_Significant'].mean():.1%}")
    print(f"归一化显著IC(p<0.05)占比: {summary_df['IC_Significant_Norm'].mean():.1%}")


if __name__ == "__main__":
    main()

从 QQQ_XGBoost_IC_Analysis/stable_features.txt 加载了 21 个稳定特征
XGBoost 稳定特征固定参数训练（含排序归一化）
发现 12 个fold
目标变量: 目标_未来10d回报率
使用稳定特征数: 21
IC计算方法: Spearman相关系数
策略: 使用稳定特征 + 固定超参数 + 排序归一化
排序归一化: 将每个Fold内预测值转换为百分位排名
固定参数:
  n_estimators: 150
  learning_rate: 0.05
  max_depth: 3
  subsample: 0.9
  colsample_bytree: 0.7
  reg_alpha: 0.5
  reg_lambda: 1.5
  min_child_weight: 1
  gamma: 0

处理 Fold 1
  使用 21 个可用稳定特征
数据规模: 训练集 (1247, 21), 测试集 (252, 21)
训练集时间范围: 2009-01-02 至 2013-12-16
测试集时间范围: 2014-01-02 至 2014-12-31
  使用固定参数训练模型...
  固定参数: {'n_estimators': 150, 'learning_rate': 0.05, 'max_depth': 3, 'subsample': 0.9, 'colsample_bytree': 0.7, 'reg_alpha': 0.5, 'reg_lambda': 1.5, 'min_child_weight': 1, 'gamma': 0, 'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1}
    原始预测值 - 训练集Spearman IC: 0.8285 (p=0.0000)
    原始预测值 - 测试集Spearman IC: 0.1199 (p=0.0573)
    归一化预测值 - 训练集Spearman IC: 0.8285 (p=0.0000)
    归一化预测值 - 测试集Spearman IC: 0.1199 (p=0.0573)
  已生成Fold 01的预测值分布图
  已生成Fold 01的散点图: QQQ_X